**강사: 멀티캠퍼스 강선구 강사** (sun9sun9@gmail.com)

## SQL 실습 노트 사용 NOTE 사용법

- 셀 실행법 I: SHIFT + 엔터: 셀을 실행 시킨후 다음 셀로 포커스를 이동
- 셀 실행법 II: CTRL + 엔터: 셀을 실행 시키고 포커스 이동 하지 않음
- 일단 아래 셀을 실행시켜주세요

In [1]:
import os
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, text
import cx_Oracle
from IPython.core.magic import register_cell_magic
import sys
if sys.version.find('GCC') >= 0:
    domain_name = 'oracle-db'
else:
    domain_name = 'localhost'
    cx_Oracle.init_oracle_client(lib_dir = r"C:\Oracle\instantclient_23_7")
DATABASE_URL = "oracle+cx_oracle://lib:multisqld@{}:1521/?service_name=XEPDB1".format(domain_name)
engine = create_engine(DATABASE_URL)
def get_rows(qry):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(qry))
            df = pd.DataFrame(result.fetchall(), columns = result.keys()).fillna('NULL').pipe(lambda x: x.set_index(x.index + 1))
            return df if len(df) != 1 else df.iloc[0]
    except sqlalchemy.exc.DatabaseError as e:
        orig = e.orig
        offset = getattr(orig, 'offset', None)
        message = getattr(orig, 'message', str(orig))
        print(message, offset)
    except Exception as e:
        print(e.__class__)
        print(e)

def exec_qrys(qrys):
    try:
        results = list()
        with engine.connect() as conn:
            for qry in qrys.split(';'):
                qry = qry.strip()
                if len(qry) == 0:
                    continue
                result = conn.execute(text(qry))
                results.append(
                    "{}, 변경된 행의 수: {}".format(qry, result.rowcount)
                )
        return results
    except Exception as e:
        print(e)

def exec_qry(qry):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(qry))
            conn.commit()
            return "변경된 행의 수: {}".format(result.rowcount)
    except Exception as e:
        print(e)

@register_cell_magic
def SQL(line, cell):
    sql = cell.strip().replace(';', '')
    if sql.lower().startswith('select'):
        return get_rows(sql)
    elif sql.lower().startswith('desc'):
        return get_rows(
"""
SELECT column_name, data_type, data_length, nullable
FROM user_tab_columns
WHERE table_name = '{}'
ORDER BY column_id
""".format(sql[5:].upper().strip())
        )
        
    else:
        exec_qry(sql)

## SQL 구문을 임의로 추가 하는 법

- "+" 버튼을 누르면 노트북에 셀이 샐깁니다.
- 셀 시작 위치에 %%SQL을 입력하고 그 다음 라인에 SQL 문을 적으면 됩니다.

In [2]:
%%SQL

SELECT * FROM tab WHERE tname not like 'BIN%' and tabtype = 'TABLE' -- 현재 database에 있는 table을 출력합니다.

,tname,tabtype,clusterid
1,BOOK,TABLE,NULL
2,BOOKCATEGORY,TABLE,NULL
3,BOOKDETAIL,TABLE,NULL
4,BOOKSTATISTIC,TABLE,NULL
5,BOOKSTOCK,TABLE,NULL
6,BOOKTAG,TABLE,NULL
7,MEMBER,TABLE,NULL
8,MEMBERSTATISTIC,TABLE,NULL
9,RATING,TABLE,NULL
10,RENT,TABLE,NULL


In [3]:
%%SQL

SELECT * FROM tab WHERE tname not like 'BIN%' and tabtype = 'VIEW' -- 현재 database에 있는 view를 출력합니다.

,tname,tabtype,clusterid
1,BOOK1,VIEW,NULL
2,BOOKSTAT,VIEW,NULL
3,BOOK_RATING,VIEW,NULL
4,MEMBER1,VIEW,NULL
5,MEMBER2,VIEW,NULL
6,RENT1,VIEW,NULL


# SELECT

## SELECT 기본 문법

```sql
SELECT [ALL/DISTINCT] 컬럼명1 [as 명칭1], 컬럼명2 [as 명칭2], ...
FROM 테이블명[ALIAS];
```

**컬럼명**: \[테이블명.\]컬럼명

**[예제 1]**  SELECT의 기본 기능을 확인합니다.


**[book1 테이블]**

In [4]:
%%SQL

DESC book1;

,column_name,data_type,data_length,nullable
1,BOOK_ID,NUMBER,22,N
2,BOOK_TITLE,VARCHAR2,300,Y
3,BOOK_SUBTITLE,VARCHAR2,300,Y
4,BOOK_CATEG_ID,NUMBER,22,Y
5,PUBLISHER,VARCHAR2,100,Y
6,AUTHORS,VARCHAR2,500,Y
7,SUMMARY,VARCHAR2,3000,Y
8,PUB_DATE,DATE,7,Y
9,REG_DATE,DATE,7,N


- book1에서 book_id, book_title을 조회합니다.

In [5]:
%%SQL

SELECT book1.book_id, book1.book_title FROM book1;

,book_id,book_title
1,2459,수상한 과학
2,1706,큰글씨 나는 이렇게 나이들고 싶다
3,1065,상실의 시대
4,1544,"궁금해요, 세종 대왕"
5,971,뉴스를 보는 눈
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관"
7,14,차근차근 배우는 우리 아이 일상 말하기
8,1384,2024 SIMPLE 의료법규
9,704,임신부·모유수유부의 안전한 약물 사용
10,3327,중학 국어 기초 완성


- 테이블명을 생략할 수 있습니다. 생략하면 FROM 구에 있는 테이블이 지정됩니다.

In [6]:
%%SQL

SELECT book_id, book_title FROM book1;

,book_id,book_title
1,2459,수상한 과학
2,1706,큰글씨 나는 이렇게 나이들고 싶다
3,1065,상실의 시대
4,1544,"궁금해요, 세종 대왕"
5,971,뉴스를 보는 눈
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관"
7,14,차근차근 배우는 우리 아이 일상 말하기
8,1384,2024 SIMPLE 의료법규
9,704,임신부·모유수유부의 안전한 약물 사용
10,3327,중학 국어 기초 완성


- book1에서 book_id의 명칭을 책 ID, book_title의 명칭을 책제목으로 하여 조회합니다.

In [7]:
%%SQL

SELECT book_id as "책 ID", book_title as 책제목
FROM book1;

,책 ID,책제목
1,2459,수상한 과학
2,1706,큰글씨 나는 이렇게 나이들고 싶다
3,1065,상실의 시대
4,1544,"궁금해요, 세종 대왕"
5,971,뉴스를 보는 눈
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관"
7,14,차근차근 배우는 우리 아이 일상 말하기
8,1384,2024 SIMPLE 의료법규
9,704,임신부·모유수유부의 안전한 약물 사용
10,3327,중학 국어 기초 완성


- book1 테이블의 alias(약칭)를 a로 하고 a에서 book_id와 book_title를 불러 옵니다. 

  book_id의 명칭은 책 ID, book_title은 책제목으로 합니다.

In [8]:
%%SQL

SELECT a.book_id as "책 ID", a.book_title as 책제목
FROM book1 a;

,책 ID,책제목
1,2459,수상한 과학
2,1706,큰글씨 나는 이렇게 나이들고 싶다
3,1065,상실의 시대
4,1544,"궁금해요, 세종 대왕"
5,971,뉴스를 보는 눈
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관"
7,14,차근차근 배우는 우리 아이 일상 말하기
8,1384,2024 SIMPLE 의료법규
9,704,임신부·모유수유부의 안전한 약물 사용
10,3327,중학 국어 기초 완성


- Asterisk * 는 모든 컬럼을 보여 줍니다.

- book1의 모든 컬럼을 가져 옵니다.

In [9]:
%%SQL

SELECT * FROM book1;

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


- book1에서 publisher 컬럼의 내용을 고유값을(중복값을 빼고) 가져 옵니다.

In [10]:
%%SQL

SELECT DISTINCT publisher FROM book1;

,publisher
1,풀빛
2,책읽는고양이
3,문학사상
4,군자출판사
5,꿈을담는틀


- book1에서 book_categ_id, publisher 의 내용을 중복 없이 가져옵니다.

In [11]:
%%SQL

SELECT DISTINCT book_categ_id, publisher FROM book1;

,book_categ_id,publisher
1,1185,풀빛
2,834,책읽는고양이
3,489,문학사상
4,731,풀빛
5,448,풀빛
6,898,책읽는고양이
7,7,군자출판사
8,654,군자출판사
9,324,군자출판사
10,1579,꿈을담는틀


## SELECT 일반적인 문법

```SQL
SELECT [ALL/DISTINCT] 표현식1 [as 명칭1], 표현식2 [as 명칭2], …
FROM 테이블명 [ALIAS];
```
### 표현식

- 리터럴, 컬럼명, 함수, 연산자를 통해 출력 컬럼 값을 정의하는 식

#### SQL 리터럴

- 리터럴: 값을 표시하는 형식

|유형|형식|예제|
|----|----|----|
|숫자(Numeric)|정수 또는 실수|123, 3.14, -42, …, 1e6(1×106)|
|문자열(String)|시작과 끝을 따옴표(') 감싼다.|'Hello', 'SQL', ….|
|날짜|	ISO 8601: 'YYYY-MM-DD'| '2025-03-04', '1977-01-13'|
|타임스탬프(날짜와시간)| 'YYYY-MM-DD hh24:mi:ss'|'2025-03-04 12:34:56'|
|NULL 값|NULL|NULL|

**[예제 2]** SQL 리터럴에 대해 알아봅니다.

- SQL 리터럴을 바탕으로 숫자, 문자열, 일자 등의 값을 나타내 봅니다

In [12]:
%%SQL

SELECT
    123 as 정수, 3.14 as 실수, 1e6 as "10의 승수", 
    'Hello' as 문자열, '2025-03-16' as 날짜, '2027-07-17 17:07:07' as 일시, NULL as 널
FROM dual;

정수                        123
실수                       3.14
10의 승수                1000000
문자열                     Hello
날짜                 2025-03-16
일시        2027-07-17 17:07:07
널                        NULL
Name: 1, dtype: object

In [13]:
%%SQL

SELECT * FROM dual;

dummy    X
Name: 1, dtype: object

※ dual 더미 테이블 - 무의미한 내용(dummy 컬럼에 X값)을 지닌 테이블로, 임의의 값을 만들고자 할 때 사용

### 연산자

#### 산술연산

|기호|설명|기호|설명|
|---|----|----|----|
|+|더하기|/|나누기|
|-|빼기|괄호()|우선순위지정|
|*|곱하기| | |

In [14]:
%%SQL

SELECT 1 + 2, 1 + 2 * 3, (1 + 2) * 3 FROM dual;

1+2        3
1+2*3      7
(1+2)*3    9
Name: 1, dtype: int64

**[예제 3]** 연산자의 기능을 알아봅니다.

- bookstat 테이블에서 rating_sum에서 rating_count를 나누어 평점평균 컬럼을 만들고,

  rent_count와 rating_count를 더해서 카운트합 컬럼을 만듭니다.

In [15]:
%%SQL

SELECT * FROM bookstat

,book_id,stat_ymd,rent_count,rating_sum,rating_count,stat_date
1,142,20220929,1,24,3,2025-09-12 04:18:07
2,3405,20240308,1,29,3,2025-09-12 04:18:07
3,2222,20220408,1,28,3,2025-09-12 04:18:07
4,2046,20250126,1,27,3,2025-09-12 04:18:07
5,2184,20221029,1,25,3,2025-09-12 04:18:07
6,1895,20230703,1,25,3,2025-09-12 04:18:07
7,1607,20210723,1,27,3,2025-09-12 04:18:07
8,1111,20230625,1,26,3,2025-09-12 04:18:07
9,270,20230213,1,28,3,2025-09-12 04:18:07


In [16]:
%%SQL

SELECT rating_sum / rating_count as 평점평균, rent_count + rating_count as 카운트합
FROM bookstat;

,평점평균,카운트합
1,8,4
2,9.66666666666666666666666666666666666667,4
3,9.33333333333333333333333333333333333333,4
4,9,4
5,8.33333333333333333333333333333333333333,4
6,8.33333333333333333333333333333333333333,4
7,9,4
8,8.66666666666666666666666666666666666667,4
9,9.33333333333333333333333333333333333333,4


#### 합성연산

문자열 결합: || 

In [17]:
%%SQL

SELECT '멀티캠퍼스' || ' - ' || 'SQLD 자격 대비 과정' as "문자열 결합" FROM dual;

문자열 결합    멀티캠퍼스 - SQLD 자격 대비 과정
Name: 1, dtype: object

**[예제 4]**  문자열 합산 연산을 사용해봅니다.

- book1 에서 {book_title} - {publish} 형식 컬럼을 출력하시오

  예) book_title: 사피엔스, publisher: 김영사 → 사피엔스 - 김영사

- || 기능을 통해 문자열을 결합해 갑니다.

In [18]:
%%SQL

SELECT book_title || ' - ' || publisher as 제목_출판사 FROM book1;

,제목_출판사
1,수상한 과학 - 풀빛
2,큰글씨 나는 이렇게 나이들고 싶다 - 책읽는고양이
3,상실의 시대 - 문학사상
4,"궁금해요, 세종 대왕 - 풀빛"
5,뉴스를 보는 눈 - 풀빛
6,"일상이 고고학, 나 혼자 국립중앙박물관 - 책읽는고양이"
7,차근차근 배우는 우리 아이 일상 말하기 - 군자출판사
8,2024 SIMPLE 의료법규 - 군자출판사
9,임신부·모유수유부의 안전한 약물 사용 - 군자출판사
10,중학 국어 기초 완성 - 꿈을담는틀


# 함수

입력을 받아 출력을 생성하는 역할자

**입력**

**수학**에서 입력을 **인자(Argument)**

**컴퓨터프로그래밍**에서 **매개변수(Parameter)**

SQL에서 함수호출법

```
함수명(매개변수1, 매개변수2, ...)
```

**[예제 5]** 함수의 사용 해봅니다.

In [19]:
%%SQL

SELECT cos(0), cos(3.1415925 / 2), sin(0), sin(3.1415925 / 2)  FROM DUAL;

COS(0)                                                       1
COS(3.1415925/2)            7.67948966192312462092172336482E-8
SIN(0)                                                       0
SIN(3.1415925/2)    0.9999999999999970512719266207883081904592
Name: 1, dtype: object

## 문자 함수
|문자 함수|기능|
|----|----|
|LOWER(문자열)|소문자로 변환|
|UPPER(문자열)|대문자로 변환|
|ASCII(문자열)|아스키 코드로 변환|
|CHR/CHAR(ASCII코드)|아스키 코드를 문자열로 변환|
|CONCAT(문자열1, 문자열2)|문자열1과 문자열2를 결합, ORACLE \|\|, SQL Server + 와 동일|
|SUBSTR/SUBSTRING(문자열, m[, n])|문자열 중에 m 위치에서 시작하여 n개의 문자들을 가져 옴.  n이 생략되면 이후 모든 문자들을 가져 옴|
|LENGTH/LEN(문자열)|길이 반환|
|LTRIM(문자열\[, 지정문자\])/LTRIM(문자열)|문자열 좌측에서 지정 문자가 나타나면 해당 문자 제거(지정 문자가 생략되면,공백)<br/>SQL Server에서는 공백이 나타나면 제거(문자 지정 불가)|
|RTRIM(문자열, 지정문자)/RTRIM(문자열)|문자열 우측에서 지정 문자가 나타나면 해당 문자 제거 (지정 문자가 생략되면,공백)<br/>SQL Server에서는 공백이 나타나면 제거(문자 지정 불가)|
|TRIM([leading\|trailing\|both] 지정문자 FROM 문자열)<br/>TRIM(지정문자 FROM 문자열)|문자열에서 머리말(좌측, leading), 꼬리말(우측, trailing), 또는 양쪽(both)에 지정 문자가 나타나면 제거. 생략 시 both <br/>SQL server는 leading, trailing, both의 지정 기능이 없고, 양쪽으로 적용됨|

**[member1 테이블]**

In [20]:
%%SQL

DESC member1;

,column_name,data_type,data_length,nullable
1,MEMBER_ID,VARCHAR2,20,N
2,MEMBER_NAME,VARCHAR2,100,Y
3,REGION,VARCHAR2,100,Y
4,ADDRESS1,VARCHAR2,100,Y
5,ADDRESS2,VARCHAR2,100,Y


**[예제 6]** 문자열 함수를 사용해봅니다.

- member1 에서 member_id와 member_id를 대분자로 변환하여 출력하시오.

In [21]:
%%SQL

SELECT member_id, upper(member_id), lower(member_id) FROM member1;

,member_id,UPPER(MEMBER_ID),LOWER(MEMBER_ID)
1,zcitby9bvw,ZCITBY9BVW,zcitby9bvw
2,9005r5vmi,9005R5VMI,9005r5vmi
3,nnvy9vtclgyfb,NNVY9VTCLGYFB,nnvy9vtclgyfb
4,n3uo4sgnjs55rx,N3UO4SGNJS55RX,n3uo4sgnjs55rx
5,q6bl2tdplxz,Q6BL2TDPLXZ,q6bl2tdplxz
6,71r05w32wx4e5j,71R05W32WX4E5J,71r05w32wx4e5j
7,osfi8te4nv9m,OSFI8TE4NV9M,osfi8te4nv9m
8,kijtmcsxqvxe4d,KIJTMCSXQVXE4D,kijtmcsxqvxe4d


- member1 에서 region과 address1를 중간에 공백을 두고 결합하고 컬럼 이름을 주소1로 하여 출력합니다.

In [22]:
%%SQL

SELECT CONCAT(CONCAT(region, ' ' ), address1) as 주소1, region || ' ' || address1 as 주소2 FROM member1;

,주소1,주소2
1,제주특별자치도 제주시,제주특별자치도 제주시
2,서울특별시 성북구,서울특별시 성북구
3,경기도 양평군,경기도 양평군
4,경상북도 포항시 북구,경상북도 포항시 북구
5,충청북도 청주시 흥덕구,충청북도 청주시 흥덕구
6,충청남도 아산시,충청남도 아산시
7,경기도 양평군,경기도 양평군
8,경상북도 포항시 남구,경상북도 포항시 남구


- member1 에서 member_name과, member_name에 앞에 첫글자 즉, 성을 떼어 내고, 그리고 두번째 글자 즉, 이름을 떼어 냅니다.

  그리고 member_name은 성명, 그 다음 성, 그리고 이름으로 컬럼명을 정합니다.

In [23]:
%%SQL

SELECT * FROM member1

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길


In [24]:
%%SQL

SELECT member_name 성명, SUBSTR(member_name, 1, 1) 성, SUBSTR(member_name, 2) 이름 FROM member1;

,성명,성,이름
1,이서진,이,서진
2,변채원,변,채원
3,최승우,최,승우
4,김연우,김,연우
5,변태진,변,태진
6,최다현,최,다현
7,권예은,권,예은
8,남채원,남,채원


- 위치 인덱스에 음수를 사용하면 마지막을 기준으로 지정할 수 있습니다.

- member1 에서 address1과 address1에서 마지막 하나 이전 부터 문자를 분리합니다.

In [25]:
%%SQL

SELECT address1, SUBSTR(address1, -1) as 구역 FROM member1;

,address1,구역
1,제주시,시
2,성북구,구
3,양평군,군
4,포항시 북구,구
5,청주시 흥덕구,구
6,아산시,시
7,양평군,군
8,포항시 남구,구


In [26]:
pd.Series([chr(i) for i in range(65, 65 + 26)], index = [i for i in range(65, 65 + 26)]).rename('').to_frame().T

,65,66,67,68,69,70,71,72,73,74,...,81,82,83,84,85,86,87,88,89,90
,A,B,C,D,E,F,G,H,I,J,...,Q,R,S,T,U,V,W,X,Y,Z


- CHR은  ASCII 코드를 문자로 치환합니다.
- ASCII는 문자를 ASCII 코드로 치환합니다.

- 문자열 코드 다루는 함수를 알아 봅니다. CHR은 ASCII 코드를 문자로, ASCII는 문자를 ASCII코드로 바꿔줍니다.

In [27]:
%%SQL

SELECT CHR(65), ASCII('A') FROM dual;

CHR(65)        A
ASCII('A')    65
Name: 1, dtype: object

- member1에서  member_id의 길이를 출력하고, member_id 

In [28]:
%%SQL

SELECT length(member_id), member_id FROM member1;

,LENGTH(MEMBER_ID),member_id
1,10,zcitby9bvw
2,9,9005r5vmi
3,13,nnvy9vtclgyfb
4,14,n3uo4sgnjs55rx
5,11,q6bl2tdplxz
6,14,71r05w32wx4e5j
7,12,osfi8te4nv9m
8,14,kijtmcsxqvxe4d


- ltrim 좌측 공백을 제거
- rtrim 우측 공백을 제거

- member1에서 member_id 앞에는 공백 2개 뒤에는 공백 3개를 붙이고, 좌측 공백을 제거하고, 우측 공백을 제거하여 문자열의 길이를 측정합니다.

In [29]:
%%SQL

SELECT length('  ' || member_id || '   '), ltrim('  ' || member_id || '   '), length(ltrim('  ' || member_id || '   ')),
ltrim('  ' || member_id || '   '), length(rtrim('  ' || member_id || '   ')) 
FROM member1;

,LENGTH(''||MEMBER_ID||''),LTRIM(''||MEMBER_ID||''),LENGTH(LTRIM(''||MEMBER_ID||'')),LTRIM(''||MEMBER_ID||''),LENGTH(RTRIM(''||MEMBER_ID||''))
1,15,zcitby9bvw,13,zcitby9bvw,12
2,14,9005r5vmi,12,9005r5vmi,11
3,18,nnvy9vtclgyfb,16,nnvy9vtclgyfb,15
4,19,n3uo4sgnjs55rx,17,n3uo4sgnjs55rx,16
5,16,q6bl2tdplxz,14,q6bl2tdplxz,13
6,19,71r05w32wx4e5j,17,71r05w32wx4e5j,16
7,17,osfi8te4nv9m,15,osfi8te4nv9m,14
8,19,kijtmcsxqvxe4d,17,kijtmcsxqvxe4d,16


- ltrim, rtrim에 공백말고 제거할 문자 지정, ltrim(문자열, 제거 문자), rtrim(문자열, 제거 문자)

1. ltrim, rtrim 기능을 확인하기 위해, member_id 좌측에 '\_\_\_' 우측에 '\_\_' 를 붙여 놓는 SQL을 만듭니다.
2. ltrim으로 좌우에 '\_'를 문자열을 붙인 것에서 좌측의 '_'를 제거합니다.
3. rtrim으로 좌우에 '\_'를 문자열을 붙인 것에서 좌측의 '_'를 제거합니다.
4. trim(both 문자열 from 컬럼)으로 양쪽에 붙은 것을 제거합니다.

In [30]:
%%SQL

SELECT '___' || member_id || '__'  FROM member1;

,'___'||MEMBER_ID||'__'
1,___zcitby9bvw__
2,___9005r5vmi__
3,___nnvy9vtclgyfb__
4,___n3uo4sgnjs55rx__
5,___q6bl2tdplxz__
6,___71r05w32wx4e5j__
7,___osfi8te4nv9m__
8,___kijtmcsxqvxe4d__


In [31]:
%%SQL

SELECT ltrim('___' || member_id || '__', '_') FROM member1;

,"LTRIM('___'||MEMBER_ID||'__','_')"
1,zcitby9bvw__
2,9005r5vmi__
3,nnvy9vtclgyfb__
4,n3uo4sgnjs55rx__
5,q6bl2tdplxz__
6,71r05w32wx4e5j__
7,osfi8te4nv9m__
8,kijtmcsxqvxe4d__


In [32]:
%%SQL

SELECT rtrim('___' || member_id || '__', '_') FROM member1;

,"RTRIM('___'||MEMBER_ID||'__','_')"
1,___zcitby9bvw
2,___9005r5vmi
3,___nnvy9vtclgyfb
4,___n3uo4sgnjs55rx
5,___q6bl2tdplxz
6,___71r05w32wx4e5j
7,___osfi8te4nv9m
8,___kijtmcsxqvxe4d


- 양쪽으로 제거할 때는 trim(both 제거할 문자 FROM 컬럼)

In [33]:
%%SQL

SELECT trim(both '_' FROM '___' || member_id || '__') FROM member1;

,TRIM(BOTH'_'FROM'___'||MEMBER_ID||'__')
1,zcitby9bvw
2,9005r5vmi
3,nnvy9vtclgyfb
4,n3uo4sgnjs55rx
5,q6bl2tdplxz
6,71r05w32wx4e5j
7,osfi8te4nv9m
8,kijtmcsxqvxe4d


## 숫자 함수 

|숫자 함수|기능|
|------|-----|
|ABS(숫자)|절대값|
|SIGN(숫자)|양수: 1, 음수: -1, 0: 0 으로 반환|
|MOD(숫자1, 숫자2)|숫자1을 숫자2로 나눌 때의 나머지 값|
|**CEIL/CEILING(숫자)**|숫자보다 크거나 같은 최소 정수|
|**FLOOR(숫자)**|숫자보다 작거나 같은 최대 정수|
|ROUND(숫자[,m])|소수점 m자리까지 반올림, 생략: m = 0|
|TRUNC(숫자[,m])|소수점 m자리 아래는 절삭. 생략: m = 0, SQL server는 미제공|

※ CEIL, FLOOR 는 헷갈립니다. 그런 만큼 시험에 잘 나오는 기능이니 꼭 기억해두세요.

**[예제 7]** 숫자 함수를 한 번씩 사용해 봅니다.

In [34]:
%%SQL


SELECT abs(-1), SIGN(5), SIGN(-5), SIGN(0), MOD(5, 3), CEIL(5.6), FLOOR(5.6), CEIL(5), FLOOR(5), ROUND(3.14159, 3), TRUNC(3.14159, 3) FROM dual

ABS(-1)                 1
SIGN(5)                 1
SIGN(-5)               -1
SIGN(0)                 0
MOD(5,3)                2
CEIL(5.6)               6
FLOOR(5.6)              5
CEIL(5)                 5
FLOOR(5)                5
ROUND(3.14159,3)    3.142
TRUNC(3.14159,3)    3.141
Name: 1, dtype: object

In [35]:
%%SQL

SELECT ceil(-2.2), floor(-2.2), trunc(-2.2), ceil(2.2), floor(2.2), trunc(2.2) FROM dual

CEIL(-2.2)    -2
FLOOR(-2.2)   -3
TRUNC(-2.2)   -2
CEIL(2.2)      3
FLOOR(2.2)     2
TRUNC(2.2)     2
Name: 1, dtype: int64

## 숫자 함수 II

|숫자 함수|기능|
|------|-----|
|SIN, COS, TAN, …|삼각함수의 값|
|EXP(숫자)|지수함수: e^숫자, e: 오일러수(2.82813…)|
|POWER(숫자1, 숫자2)|$숫자1^{숫자2}$|
|SQRT(숫자)|제곱근|
|LOG(숫자1, 숫자2)/LOG(숫자2, 숫자1)|$log_{숫자1}(숫자2)$|
|LN(숫자)|log(_e^)숫자, e: 오일러수(2.82813…)|

**[예제 8]** 숫자 함수 II를 한번 씩 사용해 봅니다.

In [36]:
%%SQL

SELECT SIN(0), COS(0), TAN(0), SIN(3.14 / 2), COS(3.14/ 2), TAN(3.14/2), EXP(2), 2.82813 * 2.82813, POWER(2, 4), SQRT(16), LOG(2, 8), LN(EXP(2)) FROM dual

SIN(0)                                                      0
COS(0)                                                      1
TAN(0)                                                      0
SIN(3.14/2)        0.9999996829318346202105299238233270001992
COS(3.14/2)          0.00079632671073332548540853364535418588
TAN(3.14/2)         1255.765591500691604660543007773873409681
EXP(2)                7.3890560989306502272304274605750078132
2.82813*2.82813                                  7.9983192969
POWER(2,4)                                                 16
SQRT(16)                                                    4
LOG(2,8)             2.99999999999999999999999999999999999999
LN(EXP(2))                                                  2
Name: 1, dtype: object

## 날짜 함수

|날짜 함수|함수 설명|
|----|-----|
|SYSDATE|현재 날짜와 시각을 구함|
|EXTRACT(YEAR\|MONTH\|DAY from d)|날짜 데이터에서 연월일 데이터를 추출|

**[bookstat 테이블]**

In [37]:
%%SQL

DESC bookstat;

,column_name,data_type,data_length,nullable
1,BOOK_ID,NUMBER,22,N
2,STAT_YMD,VARCHAR2,8,N
3,RENT_COUNT,NUMBER,22,Y
4,RATING_SUM,NUMBER,22,Y
5,RATING_COUNT,NUMBER,22,Y
6,STAT_DATE,DATE,7,N


**[예제 9]** 날짜 함수를 사용해봅니다.

- sysdate와 bookstat 에서 stat_date를 불러오고 sysdate와 stat_date의 일차를 구하고, 소수 둘째 자리까지 반올림합니다.`

In [38]:
%%SQL

SELECT sysdate, stat_date, sysdate - stat_date, round(sysdate - stat_date, 2) FROM bookstat;

,sysdate,stat_date,SYSDATE-STAT_DATE,"ROUND(SYSDATE-STAT_DATE,2)"
1,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23
2,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23
3,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23
4,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23
5,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23
6,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23
7,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23
8,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23
9,2026-05-15 09:54:10,2025-09-12 04:18:07,245.233368055555555555555555555555555556,245.23


- stat_date와 stat_date에서 연도, 월, 일 을 가져옵니다. 

In [39]:
%%SQL

SELECT stat_date, extract(YEAR from stat_date) as 연, extract(MONTH from stat_date) as 월, extract(DAY from stat_date) as 일 FROM bookstat;

,stat_date,연,월,일
1,2025-09-12 04:18:07,2025,9,12
2,2025-09-12 04:18:07,2025,9,12
3,2025-09-12 04:18:07,2025,9,12
4,2025-09-12 04:18:07,2025,9,12
5,2025-09-12 04:18:07,2025,9,12
6,2025-09-12 04:18:07,2025,9,12
7,2025-09-12 04:18:07,2025,9,12
8,2025-09-12 04:18:07,2025,9,12
9,2025-09-12 04:18:07,2025,9,12


## 형변환 함수

|형변환 함수|설명|
|-----|----|
|TO_NUMBER(문자형)|문자형 데이터를 숫자형으로 변환|
|TO_CHAR(숫자\|날짜\[, 포맷\])|수치/날짜형 데이터를 문자형으로 변환|
|TO_DATE(문자형\[, 포맷\])|문자형 데이터를 날짜형으로 변환|



**[예제 10]**  형변환 함수를 사용해봅니다.

In [40]:
%%SQL

SELECT stat_ymd, substr(stat_ymd, 1, 4) as 문자형연도, substr(stat_ymd, 5, 2) as 문자형월, substr(stat_ymd, 7, 2) as 문자형일,
    substr(stat_ymd, 1, 4) || substr(stat_ymd, 5, 2) || substr(stat_ymd, 7, 2) as 재결합,
    substr(stat_ymd, 1, 4) + substr(stat_ymd, 5, 2) + substr(stat_ymd, 7, 2) as 합, --암시적 데이터 유형 변환: 문자형 -> 숫자형
    to_number(substr(stat_ymd, 1, 4)) as 숫자형연도, to_number(substr(stat_ymd, 5, 2)) as 숫자형월, to_number(substr(stat_ymd, 7, 2)) as 숫자형일,
    to_number(substr(stat_ymd, 1, 4)) || to_number(substr(stat_ymd, 5, 2)) || to_number(substr(stat_ymd, 7, 2)) as 재결합2 --암시적 데이터 유형 변환: 숫자형 -> 문자형
FROM bookstat

,stat_ymd,문자형연도,문자형월,문자형일,재결합,합,숫자형연도,숫자형월,숫자형일,재결합2
1,20220929,2022,09,29,20220929,2060,2022,9,29,2022929
2,20240308,2024,03,08,20240308,2035,2024,3,8,202438
3,20220408,2022,04,08,20220408,2034,2022,4,8,202248
4,20250126,2025,01,26,20250126,2052,2025,1,26,2025126
5,20221029,2022,10,29,20221029,2061,2022,10,29,20221029
6,20230703,2023,07,03,20230703,2033,2023,7,3,202373
7,20210723,2021,07,23,20210723,2051,2021,7,23,2021723
8,20230625,2023,06,25,20230625,2054,2023,6,25,2023625
9,20230213,2023,02,13,20230213,2038,2023,2,13,2023213


- 일자형식
| 형식 문자열 | 설명                         | 예시 (`2025-08-07 14:30:45`) |
|-------------|------------------------------|-----------------------------|
| `YYYY`      | 4자리 연도                   | `2025`                      |
| `YY`        | 2자리 연도                   | `25`                        |
| `MM`        | 월 (2자리 숫자)              | `08`                        |
| `MON`       | 월 (영문 약어)               | `AUG`                       |
| `MONTH`     | 월 (영문 전체, 공백 포함)    | `AUGUST   `                 |
| `DD`        | 일 (2자리 숫자)              | `07`                        |
| `DY`        | 요일 (영문 약어)             | `THU`                       |
| `DAY`       | 요일 (영문 전체, 공백 포함)  | `THURSDAY `                 |
| `HH24`      | 시간 (24시간제)              | `14`                        |
| `MI`        | 분                           | `30`                        |
| `SS`        | 초                           | `45`                        |
| `AM` / `PM` | 오전/오후 표시 (12시간제)    | `PM`                        |
| `RR`        | 세기 보정 포함 2자리 연도    | `25`                        |
| `YYYYMMDD`  | 연월일 압축 포맷             | `20250807`                  |
| `J`         | 줄리안 날짜                  | `2460536`                   |

**[예제 11]** 일자형 형식을 사용하여 형변환을 합니다.

- stat_ymd를 일자 형식으로 변환합니다.

In [41]:
%%SQL

SELECT stat_ymd, to_date(stat_ymd, 'YYYYMMDD'), to_date(stat_ymd, 'YYYYMMDD') + 1 / 24 FROM bookstat

,stat_ymd,"TO_DATE(STAT_YMD,'YYYYMMDD')","TO_DATE(STAT_YMD,'YYYYMMDD')+1/24"
1,20220929,2022-09-29,2022-09-29 01:00:00
2,20240308,2024-03-08,2024-03-08 01:00:00
3,20220408,2022-04-08,2022-04-08 01:00:00
4,20250126,2025-01-26,2025-01-26 01:00:00
5,20221029,2022-10-29,2022-10-29 01:00:00
6,20230703,2023-07-03,2023-07-03 01:00:00
7,20210723,2021-07-23,2021-07-23 01:00:00
8,20230625,2023-06-25,2023-06-25 01:00:00
9,20230213,2023-02-13,2023-02-13 01:00:00


- stat_date를 일자 형식 '연도4자리/월2자리/일2자리'에 맞춰 문자열로 변환합니다.

In [42]:
%%SQL

SELECT stat_date, to_char(stat_date, 'yyyy/mm/dd') FROM bookstat

,stat_date,"TO_CHAR(STAT_DATE,'YYYY/MM/DD')"
1,2025-09-12 04:18:07,2025/09/12
2,2025-09-12 04:18:07,2025/09/12
3,2025-09-12 04:18:07,2025/09/12
4,2025-09-12 04:18:07,2025/09/12
5,2025-09-12 04:18:07,2025/09/12
6,2025-09-12 04:18:07,2025/09/12
7,2025-09-12 04:18:07,2025/09/12
8,2025-09-12 04:18:07,2025/09/12
9,2025-09-12 04:18:07,2025/09/12


|함수|함수 설명|
|---|-----|
|NVL(표현식1, 표현식2) / ISNULL(표현식1, 표현식2)}|표현식1의 결과값이 NULL이면, 표현식2의 결과값으로 출력|
|NULLIF(표현식1, 표현식2)|표현식1의 결과값과 표현식2의 결과값이 같으면 NULL 아니면, 표현식1의 결과값으로 출력|
|COALESCE(표현식1, 표현식2, …, 표현식n)|표현식1에서 표현식n까지 결과값이 최초로 NULL아닌 것을 출력|


- NVL은 표현식1의 결과 중 NULL 값을 표현식2의 값으로 채웁니다.

**[예제 12]** NULL 값처리 함수를 알아봅니다.

- return_date에는 NULL값이 있습니다. NULL 값일 때는 1900-01-01으로 채웁니다.

**[rent1 테이블]**

In [43]:
%%SQL

DESC rent1;

,column_name,data_type,data_length,nullable
1,BOOK_ID,NUMBER,22,Y
2,STOCK_SEQ,NUMBER,22,Y
3,MEMBER_ID,VARCHAR2,20,Y
4,RENT_NO,NUMBER,22,Y
5,RENT_DATE,DATE,7,Y
6,RETURN_DATE,DATE,7,Y


In [44]:
%%SQL

SELECT return_date, NVL(return_date, to_date('1900-01-01', 'yyyy-mm-dd')) FROM rent1

,return_date,"NVL(RETURN_DATE,TO_DATE('1900-01-01','YYYY-MM-DD'))"
1,NULL,1900-01-01 00:00:00
2,NULL,1900-01-01 00:00:00
3,2025-02-24 15:20:23,2025-02-24 15:20:23
4,NULL,1900-01-01 00:00:00
5,NULL,1900-01-01 00:00:00
6,NULL,1900-01-01 00:00:00
7,2025-02-21 12:47:44,2025-02-21 12:47:44
8,2025-02-28 16:28:21,2025-02-28 16:28:21
9,2025-02-25 10:42:32,2025-02-25 10:42:32
10,2025-02-28 12:50:36,2025-02-28 12:50:36


- stock_seq가 1 이면 NULL 이 되게 합니다. 그리고 앞의 연산 결과를 NVL을 사용해서 NULL인 값을 1000으로 채웁니다.

In [45]:
%%SQL

SELECT stock_seq, NULLIF(stock_seq, 1), NVL(NULLIF(stock_seq, 1), 1000) FROM rent1

,stock_seq,"NULLIF(STOCK_SEQ,1)","NVL(NULLIF(STOCK_SEQ,1),1000)"
1,2,2.0,2
2,1,NULL,1000
3,2,2.0,2
4,3,3.0,3
5,1,NULL,1000
6,1,NULL,1000
7,2,2.0,2
8,1,NULL,1000
9,2,2.0,2
10,1,NULL,1000


- return_date가 NULL인 경우를 rent_date로 이 값도 NULL이면 1900-01-01 일로 채워 출력합니다.

In [46]:
%%SQL

SELECT return_date, rent_date, 
    COALESCE(return_date, rent_date, to_date('1900-01-01', 'yyyy-mm-dd')) FROM rent1

,return_date,rent_date,"COALESCE(RETURN_DATE,RENT_DATE,TO_DATE('1900-01-01','YYYY-MM-DD'))"
1,NULL,NULL,1900-01-01 00:00:00
2,NULL,2025-02-20 16:52:29,2025-02-20 16:52:29
3,2025-02-24 15:20:23,2025-02-20 16:43:29,2025-02-24 15:20:23
4,NULL,NULL,1900-01-01 00:00:00
5,NULL,2025-02-20 16:52:14,2025-02-20 16:52:14
6,NULL,2025-02-20 16:56:31,2025-02-20 16:56:31
7,2025-02-21 12:47:44,2025-02-20 16:45:01,2025-02-21 12:47:44
8,2025-02-28 16:28:21,2025-02-20 16:42:31,2025-02-28 16:28:21
9,2025-02-25 10:42:32,2025-02-20 16:51:15,2025-02-25 10:42:32
10,2025-02-28 12:50:36,2025-02-20 16:46:03,2025-02-28 12:50:36


# Case 문

```sql
Case
    WHEN 조건식1 THEN 표현식1
    [WHEN 조건식2 THEN 표현식2
    ...
    WHEN 조건식n THEN 표현식n]
    ELSE 기본 표현식
END
```

## 비교 연산 I

|비교 연산|의미|
|---|---|
|표현식1 = 표현식2|표현식1의 결과값이 표현식2의 결과값과 같은가?|
|표현식1 <> 표현식2,<br/> 표현식1 != 표현식2|표현식1의 결과값이 표현식2의 결과값과 다른가?|
|표현식1 > 표현식2|표현식1의 결과값이 표현식2의 결과값보다 큰가?|
|표현식1 < 표현식2|표현식1의 결과값이 표현식2의 결과값보다 작은가?|
|표현식1 >= 표현식2|표현식1의 결과값이 표현식2의 결과값보다 같거나 큰가?|
|표현식1 <= 표현식2|표현식1의 결과값이 표현식2의 결과값보다 같거나 작은가?|


**[book_rating 테이블]**

In [47]:
%%SQL

DESC book_rating;

,column_name,data_type,data_length,nullable
1,BOOK_ID,NUMBER,22,N
2,RATING,NUMBER,22,Y


In [48]:
%%SQL

SELECT * FROM book_rating -- book_rating을 불러 옵니다.

,book_id,rating
1,142,4
2,152,4.17
3,1111,4.33
4,1607,4.5
5,1895,4.17
6,2184,4.17
7,2046,4.5


**[예제 13]** 
book_rating에서 아래 내용을 가져옵니다.

| 컬럼 이름 | 설명                                                                 |
|-----------|----------------------------------------------------------------------|
| book_id   | 도서의 고유 식별자. 각 레코드를 유일하게 구분하는 기본 키 역할을 합니다.         |
| rating    | 도서에 대한 평점 값. 실수형 숫자로 표현되며, 0 이상 5 이하의 값입니다.             |
| star      | rating 값에 따라 다음 기준으로 환산된 별점 문자열입니다:                        |
|           | - rating ≥ 4.5     → ★★★★★                                             |
|           | - 4.25 ≤ rating < 4.5 → ★★★★☆                                          |
|           | - 4.0 ≤ rating < 4.25 → ★★★☆☆                                          |
|           | - rating < 4.0       → ★★☆☆☆                                           |

In [49]:
%%SQL

SELECT book_id, rating,
    CASE
        WHEN rating >= 4.5 THEN '★★★★★'
        WHEN rating >= 4.25 THEN '★★★★☆'
        WHEN rating >= 4 THEN '★★★☆☆'
        ELSE '★★☆☆☆'
    END as star
FROM book_rating

,book_id,rating,star
1,142,4,★★★☆☆
2,152,4.17,★★★☆☆
3,1111,4.33,★★★★☆
4,1607,4.5,★★★★★
5,1895,4.17,★★★☆☆
6,2184,4.17,★★★☆☆
7,2046,4.5,★★★★★


## 비교 연산 II

|비교 연산자|의미|
|----|----|
|BETWEEN a AND b|a와 b 사이의 값인가? (a, b 포함)|
|IN (list)|list 에 있는 값인가?|
|LIKE '비교문자열'|비교문자열에 해당하는가?<br/>※ 비교문자열: %, _ 기호를 통한 ‘\~시작하는’, ‘\~로 끝나는’ 등의 의미를 표현|
|IS NULL|NULL 값인가? <br/>NULL 값과 일반 비교 연산은 항상 FALSE|


**[예제 14]** 비교 연산을 활용해 봅니다.

**[member2 테이블]**

In [50]:
%%SQL

DESC member2;

,column_name,data_type,data_length,nullable
1,MEMBER_ID,VARCHAR2,20,N
2,REGION,VARCHAR2,100,Y
3,ADDRESS1,VARCHAR2,100,Y
4,BIRTH_YEAR,NUMBER,22,N


In [51]:
%%SQL

SELECT * FROM member2 -- member2의 내용을 봅니다.

,member_id,region,address1,birth_year
1,zcitby9bvw,제주특별자치도,제주시,1974
2,9005r5vmi,서울특별시,성북구,1974
3,nnvy9vtclgyfb,경기도,양평군,1990
4,n3uo4sgnjs55rx,경상북도,포항시 북구,2010
5,q6bl2tdplxz,충청북도,청주시 흥덕구,1973
6,71r05w32wx4e5j,충청남도,아산시,1975
7,osfi8te4nv9m,경기도,양평군,1978


- member2에서 아래 내용을 가져옵니다.

| 컬럼명     | 설명                                                                 |
|---------------|----------------------------------------------------------------------|
| birth_year    | 회원의 출생 연도. 숫자 형식이며, 나이 계산을 위해 현재 연도와의 차이로 사용됩니다. |
| student_type  | 출생 연도 기준 나이(age)를 활용해 구분한 학생 유형. 구분 기준은 아래와 같습니다: |
|               | - 나이 < 8           → '유치원생'                                     |
|               | - 8 ≤ 나이 ≤ 13      → '초등학생'                                     |
|               | - 14 ≤ 나이 ≤ 16     → '중학생'                                       |
|               | - 17 ≤ 나이 ≤ 19     → '고등학생'                                     |
|               | - 나이 > 19          → '대학생'                                       |


In [52]:
%%SQL

SELECT birth_year as 출생연도, extract(YEAR from sysdate) - birth_year as 나이, 
    case 
        when extract(YEAR from sysdate) - birth_year < 8 THEN '유치원생'
        when extract(YEAR from sysdate) - birth_year BETWEEN 8 AND 13 THEN '초등학생'
        when extract(YEAR from sysdate) - birth_year BETWEEN 14 AND 16 THEN '중학생'
        when extract(YEAR from sysdate) - birth_year BETWEEN 17 AND 19 THEN '고등학생'
        else '대학생'
    end as 학생구분
FROM member2

,출생연도,나이,학생구분
1,1974,52,대학생
2,1974,52,대학생
3,1990,36,대학생
4,2010,16,중학생
5,1973,53,대학생
6,1975,51,대학생
7,1978,48,대학생


-  member2에서 아래 내용을 가져옵니다.

| 컬럼명 | 설명                                                                 |
|-----------|----------------------------------------------------------------------|
| address1  | 회원의 기본 주소 정보. 시/군/구 등의 지명 정보가 포함됩니다.             |
| region2   | address1의 마지막 글자를 기준으로 구분한 지역 유형. 구분 기준은 아래와 같습니다: |
|           | - address1이 '시'로 끝나면     → '시'                                    |
|           | - address1이 '군'으로 끝나면   → '군'                                    |
|           | - 위의 조건에 해당하지 않으면 → '기타'                                  |


In [53]:
%%SQL

SELECT address1, 
    case 
        when address1 LIKE '%시' then '시'
        when address1 LIKE '%군' then '군'
        else '기타'
    end as region2
FROM member2

,address1,region2
1,제주시,시
2,성북구,기타
3,양평군,군
4,포항시 북구,기타
5,청주시 흥덕구,기타
6,아산시,시
7,양평군,군


- member2에서 아래 내용을 가져옵니다.

| 컬럼명| 설명                                                                 |
|-----------|----------------------------------------------------------------------|
| region    | 회원의 지역 정보를 문자열의 시작 패턴에 따라 분류한 지역 구분 컬럼입니다. 구분 기준은 다음과 같습니다: |
|           | - region 값이 '경상'으로 시작하면 → '경상도'                           |
|           | - region 값이 '전라'로 시작하면   → '전라도'                           |
|           | - region 값이 '충청'으로 시작하면 → '충청도'                           |
|           | - 위 조건에 해당하지 않으면      → 기존 region 값을 그대로 출력 또는 별도 처리 |


In [54]:
%%SQL

SELECT region, 
    case 
        when region LIKE '경상%' then '경상도'
        when region LIKE '전라%' then '전라도'
        when region LIKE '충청%' then '충청도'
        else region
    end as region2
FROM member2

,region,region2
1,제주특별자치도,제주특별자치도
2,서울특별시,서울특별시
3,경기도,경기도
4,경상북도,경상도
5,충청북도,충청도
6,충청남도,충청도
7,경기도,경기도


In [55]:
%%SQL

SELECT * FROM member2 WHERE address1 LIKE '__'

,member_id,region,address1,birth_year


- book1에서 아래 내용을 가져옵니다.

| 컬럼 이름 | 설명                                                                 |
|-----------|------------------------------------------------------------------------------|
| 요일      | pub_date 컬럼을 요일(Day) 형식으로 문자 변환한 값. 예: 'Monday', 'Tuesday' 등  |
| 일자구분  | 요일이 'Sunday' 또는 'Saturday'이면 '휴일', 그렇지 않으면 '평일'로 구분한 값     |


In [56]:
%%SQL

SELECT to_char(pub_date, 'Day') as 요일,
    case 
        when trim(to_char(pub_date, 'Day')) in ('Sunday', 'Saturday') then '휴일'
        else '평일'
    end 일자구분
FROM book1

,요일,일자구분
1,Thursday,평일
2,Thursday,평일
3,Sunday,휴일
4,Monday,평일
5,Tuesday,평일
6,Friday,평일
7,Wednesday,평일
8,Monday,평일
9,Friday,평일
10,Tuesday,평일


- NULL은 '비어 있음'을 의미합니다. 비교문에서 '=', '<>', '>=', '<=' 비교연산 결과 모두 Unknown 입니다.

- rent에서 아래 내용을 가져 옵니다.

| 컬럼명 | 설명                                                                 |
|-----------|------------------------------------------------------------------------------|
| rent_date | 대여 일자. rent1 테이블의 원본 날짜 값입니다.                               |
| NULL1     | `rent_date = NULL` 조건으로 처리한 연도 추출. 항상 else절만 실행되어 연도만 출력됩니다. *(잘못된 NULL 비교 방식)* |
| NULL2     | `rent_date`가 NULL이면 -1, NULL이 아니면 해당 날짜의 연도(`extract(YEAR from rent_date)`)를 출력합니다. |


In [57]:
%%SQL

SELECT rent_date, 
    case 
        when rent_date = NULL then -1
        else extract(YEAR from rent_date)
    end as NULL1,
    case
        when rent_date is NULL then -1
        else extract(YEAR from rent_date)
    end as NULL2
FROM rent1

,rent_date,null1,null2
1,NULL,NULL,-1
2,2025-02-20 16:52:29,2025.0,2025
3,2025-02-20 16:43:29,2025.0,2025
4,NULL,NULL,-1
5,2025-02-20 16:52:14,2025.0,2025
6,2025-02-20 16:56:31,2025.0,2025
7,2025-02-20 16:45:01,2025.0,2025
8,2025-02-20 16:42:31,2025.0,2025
9,2025-02-20 16:51:15,2025.0,2025
10,2025-02-20 16:46:03,2025.0,2025


## 논리 연산자

|논리 연산자|의미|
|----|----|
|조건식1 AND 조건식2|조건식1과 조건식2를 만족하는가?|
|조건식1 OR 조건식2|조건식1또는 조건식2를 만족하는가?
|NOT 조건식|조건식을 만족하지 않는가?|


**[예제 15]** book1에서 아래 내용을 가져 옵니다.

| 컬럼명   | 설명 |
|----------|------|
| pub_date | `book1` 테이블의 도서 발행일을 나타내는 컬럼입니다. 원본 날짜 데이터를 그대로 출력합니다. |
| 월       | `extract(MONTH from pub_date)` 함수를 사용하여 발행일에서 월(Month) 숫자(1~12)를 추출합니다. 이후 계절 분류의 기준 값으로 사용됩니다. |
| 계절     | `pub_date`에서 추출한 월 값을 기준으로 다음 조건을 이용해 계절을 분류합니다:<br> - `월 ≥ 12 or 월 ≤ 2` → `'겨울'`<br> - `월 ≥ 3 and 월 ≤ 5` → `'봄'`<br> - `월 ≥ 6 and 월 ≤ 8` → `'여름'`<br> - 그 외 (`9~11월`) → `'가을'`<br> `CASE WHEN` 구문에서 `OR`, `AND`, `>=`, `<=`을 이용한 조건 분기 방식 예시입니다. |
| 계절2    | 동일한 계절 분류 작업을 `BETWEEN`, `NOT BETWEEN`을 활용하여 처리한 컬럼입니다. 조건은 다음과 같습니다:<br> - `월 NOT BETWEEN 3 AND 11` → `'겨울'`<br> - `월 BETWEEN 3 AND 5` → `'봄'`<br> - `월 BETWEEN 6 AND 8` → `'여름'`<br> - 그 외 (`9~11월`) → `'가을'`<br> 같은 결과를 다른 SQL 문법으로 구현한 예시로, 조건 표현 방식 비교 학습에 유용합니다. |


In [58]:
%%SQL

SELECT pub_date, extract(MONTH from pub_date) as 월,
    CASE 
        WHEN extract(MONTH from pub_date) >= 12 or extract(MONTH from pub_date) <= 2 THEN '겨울'
        WHEN extract(MONTH from pub_date) >= 3 AND extract(MONTH from pub_date) <= 5 THEN '봄'
        WHEN extract(MONTH from pub_date) >= 6 AND  extract(MONTH from pub_date) <= 8 THEN '여름'
        ELSE '가을'
    END as 계절,
    CASE 
        WHEN NOT extract(MONTH from pub_date) BETWEEN 3 AND 11 THEN '겨울'
        WHEN extract(MONTH from pub_date) BETWEEN 3 AND 5 THEN '봄'
        WHEN extract(MONTH from pub_date) BETWEEN 6 AND 8 THEN '여름'
        ELSE '가을'
    END as 계절2
FROM book1

,pub_date,월,계절,계절2
1,2004-01-01,1,겨울,겨울
2,2004-07-01,7,여름,여름
3,2000-10-01,10,가을,가을
4,2019-07-01,7,여름,여름
5,2019-10-01,10,가을,가을
6,2022-07-01,7,여름,여름
7,2023-02-01,2,겨울,겨울
8,2023-05-01,5,봄,봄
9,2024-11-01,11,가을,가을
10,2024-10-01,10,가을,가을


## 단순 Case 문


```sql
Case 표현식
    WHEN 조건값1 THEN 표현식1
    [WHEN 조건값2 THEN 표현식2
    ...
    WHEN 조건값n THEN 표현식n]
    ELSE 기본 표현식
END
```

**[예제 16]** book1에서 아래 내용을 가져 옵니다.

| 컬럼명   | 설명 |
|----------|------|
| pub_date | `book1` 테이블의 도서 발행일을 나타내는 컬럼입니다. 원본 날짜(Date) 데이터를 그대로 출력합니다. |
| weekday  | `to_char(pub_date, 'DAY')` 함수를 사용하여 발행일을 요일 문자열로 변환한 값입니다. 결과는 영어 대문자로 반환되며, 요일 문자열 끝에는 공백이 포함될 수 있어 `trim()` 처리가 필요합니다. |
| 요일     | `trim(to_char(pub_date, 'DAY'))` 결과 값을 기준으로 `CASE` 문을 이용해 해당 요일을 **한글 요일명**으로 매핑한 컬럼입니다. 조건은 다음과 같습니다:<br> - `'SUNDAY'` → `'일요일'`<br> - `'MONDAY'` → `'월요일'`<br> - `'TUESDAY'` → `'화요일'`<br> - `'WEDNESDAY'` → `'수요일'`<br> - `'THURSDAY'` → `'목요일'`<br> - `'FRIDAY'` → `'금요일'`<br> - 그 외 (예: `'SATURDAY'`) → `'토요일'`<br> 영문 요일을 한글로 변환하는 텍스트 매핑 처리 Task 예시입니다. |


In [59]:
%%SQL

SELECT pub_date, to_char(pub_date, 'DAY') as weekday,
case trim(to_char(pub_date, 'DAY'))
    when 'SUNDAY' then '일요일'
    when 'MONDAY' then '월요일'
    when 'TUESDAY' then '화요일'
    when 'WEDNESDAY' then '수요일'
    when 'THURSDAY' then '목요일'
    when 'FRIDAY' then '금요일'
    else '토요일'
    end as 요일
FROM book1

,pub_date,weekday,요일
1,2004-01-01,THURSDAY,목요일
2,2004-07-01,THURSDAY,목요일
3,2000-10-01,SUNDAY,일요일
4,2019-07-01,MONDAY,월요일
5,2019-10-01,TUESDAY,화요일
6,2022-07-01,FRIDAY,금요일
7,2023-02-01,WEDNESDAY,수요일
8,2023-05-01,MONDAY,월요일
9,2024-11-01,FRIDAY,금요일
10,2024-10-01,TUESDAY,화요일


# WHERE 절

```SQL
SELECT [ALL/DISTINCT] 표현식1 [as 명칭1], 표현식2 [as 명칭2], …
FROM 테이블명 [ALIAS]
[WHERE 조건식];
```

**[예제 17]** WHERE 조건문을 활용해봅니다.

- member1에서 member_name 이 '준'으로 끝나는 행을 가져옵니다.

In [60]:
%%SQL

SELECT * FROM member1 
WHERE member_name like '%준'

,member_id,member_name,region,address1,address2


- member1에서 member_name 이 '박'으로 시작하는 행을 가져옵니다.

In [61]:
%%SQL

SELECT * FROM member1 
WHERE member_name like '박%'

,member_id,member_name,region,address1,address2


- member1에서 member_name 에 '기'가 들어가는 행을 가져오세요.

In [62]:
%%SQL

SELECT * FROM member1 
WHERE member_name like '%기%'

,member_id,member_name,region,address1,address2


- member1  에서 LIKE  member_name '__' 통한 두 글자 이름을 가져와 봅니다. 

In [63]:
%%SQL

SELECT * FROM member1 WHERE member_name like '__'

,member_id,member_name,region,address1,address2


- book1에서 pub_date의 월이 4에서 6사이인 행을 가져 옵니다.

In [64]:
%%SQL

SELECT pub_date, extract(MONTH FROM pub_date), 
    to_number(to_char(pub_date, 'mm')) as mon -- projection  last 
FROM book1 -- 1
WHERE mon BETWEEN 4 and 6 -- selection 2

ORA-00904: "MON": invalid identifier None


In [65]:
%%SQL

SELECT * FROM book1 
WHERE extract(MONTH from pub_date) between 4 AND 6

book_id                                                       1384
book_title                                        2024 SIMPLE 의료법규
book_subtitle                                                 NULL
book_categ_id                                                  654
publisher                                                    군자출판사
authors                                                         길벗
summary          “법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...
pub_date                                       2023-05-01 00:00:00
reg_date                                       2023-05-01 12:38:16
Name: 1, dtype: object

- book1에서 publisher가 '마더텅'이고 pubdate가 2019-01-01 일 이상인 행을 가져옵니다.

In [66]:
%%SQL

SELECT * FROM book1 
WHERE publisher = '풀빛' AND pub_date >= to_date('2019-01-01', 'yyyy-mm-dd')

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
2,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43


In [67]:
%%SQL

SELECT * FROM book1 
WHERE publisher = '풀빛' AND to_char(pub_date, 'yyyy-mm-dd') >= '2019-01-01'

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
2,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43


- rent1에서 'rent_date = NULL'인 조건으로 행을 가져옵니다.

In [68]:
%%SQL

SELECT * FROM rent1 WHERE rent_date = NULL

,book_id,stock_seq,member_id,rent_no,rent_date,return_date


- rent1에서 rent__date가 NULL인 행을 가져 옵니다.

In [69]:
%%SQL

SELECT * FROM rent1 WHERE rent_date is NULL

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,2243,2,1jhmhlrlo71brwi4o8,1,NULL,NULL
2,3194,3,94zfx2tf92heh59z,1,NULL,NULL


- rent1에서 'rent_date <> NULL'인 조건으로 행을 가져옵니다.

In [70]:
%%SQL

SELECT * FROM rent1 WHERE rent_date <> NULL

,book_id,stock_seq,member_id,rent_no,rent_date,return_date


- rent1에서 rent_date  NULL이 아닌 행을 가져옵니다.

In [71]:
%%SQL

SELECT * FROM rent1 WHERE rent_date is not NULL

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,970,1,46vpd6v91zb01e,1,2025-02-20 16:52:29,NULL
2,2403,2,4e1dh553s37,1,2025-02-20 16:43:29,2025-02-24 15:20:23
3,299,1,haejt93chfpl,1,2025-02-20 16:52:14,NULL
4,2239,1,ll65teqf4kzep8l1r,1,2025-02-20 16:56:31,NULL
5,2180,2,nrz2bnr8s6,1,2025-02-20 16:45:01,2025-02-21 12:47:44
6,862,1,r1cgy5kq5bviueq5,1,2025-02-20 16:42:31,2025-02-28 16:28:21
7,3251,2,uce50kz7so3,1,2025-02-20 16:51:15,2025-02-25 10:42:32
8,2364,1,uck1pr4ofirj2a513,1,2025-02-20 16:46:03,2025-02-28 12:50:36


# GROUP BY

```SQL
SELECT [DISTINCT] 표현식1 [as 명칭1][, 표현식2 [as 명칭2], …]
FROM 테이블명 [ALIAS]
[WHERE 조건식]
[GROUP BY 표현식1[, 표현식2, …]]
```

- GROUP BY 표현식에 표현식에 해당하는 그룹핑 값만 선택하며 그룹 정보만 선택이 됩니다. 즉 그룹의 고유값들만 출력합니다.

**[예제 18]**  그룹핑만을 합니다.

1. book1에서 publisher로 그룹핑을 합니다.

In [72]:
%%SQL 

SELECT publisher FROM book1

,publisher
1,풀빛
2,책읽는고양이
3,문학사상
4,풀빛
5,풀빛
6,책읽는고양이
7,군자출판사
8,군자출판사
9,군자출판사
10,꿈을담는틀


In [73]:
%%SQL 

SELECT publisher FROM book1
GROUP BY publisher

,publisher
1,풀빛
2,책읽는고양이
3,문학사상
4,군자출판사
5,꿈을담는틀


2. 함수를 사용할 수도 있습니다. book1에서 pub_date에서 연도를 그룹핑합니다.

In [74]:
%%SQL 

SELECT extract(YEAR from pub_date) as year
FROM book1
GROUP BY extract(YEAR from pub_date)

,year
1,2004
2,2000
3,2019
4,2022
5,2023
6,2024


In [75]:
%%SQL

SELECT extract(YEAR from pub_date) as year -- last
FROM book1 -- 1
WHERE extract(YEAR from pub_date) <= 2000 -- 2
GROUP BY extract(YEAR from pub_date) -- 3

year    2000
Name: 1, dtype: int64

- WHERE 조건문은 FROM과 GROUP BY 사이에 옵니다

3. book1에서 pub_date에서 연도를 그룹핑합니다. 그리고, 연도가 2000년 이전의 행만을 뽑습니다. 

In [76]:
%%SQL

SELECT extract(YEAR from pub_date) as year
FROM book1
WHERE extract(YEAR from pub_date) <= 2000
GROUP BY extract(YEAR from pub_date)

year    2000
Name: 1, dtype: int64

## 집계함수

여러 행의 결과를 집계하여 하나의 결과 값으로 만들어 주는 함수

|집계함수|의미|
|------|------|
|COUNT(*)|NULL 값을 포함한 행의 수|
|COUNT([DISTINCT\|ALL] 표현식)|NULL을 제외한 행의 수|
|SUM([DISTINCT\|ALL] 표현식)|NULL을 제외한 합계|
|AVG([DISTINCT\|ALL] 표현식)|NULL을 제외한 평균|
|MAX([DISTINCT\|ALL] 표현식)|최대값|
|MIN([DISTINCT\|ALL] 표현식)|최소값|
|STDDEV([DISTINCT\|ALL] 표현식)|NULL을 제외한 표준 편차|
|VARIANCE/VAR([DISTINCT\|ALL] 표현식)|NULL을 제외한 분산|

In [77]:
%%SQL

SELECT * FROM rent1 -- rent1의 내용을 봅니다.

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,2243,2,1jhmhlrlo71brwi4o8,1,NULL,NULL
2,970,1,46vpd6v91zb01e,1,2025-02-20 16:52:29,NULL
3,2403,2,4e1dh553s37,1,2025-02-20 16:43:29,2025-02-24 15:20:23
4,3194,3,94zfx2tf92heh59z,1,NULL,NULL
5,299,1,haejt93chfpl,1,2025-02-20 16:52:14,NULL
6,2239,1,ll65teqf4kzep8l1r,1,2025-02-20 16:56:31,NULL
7,2180,2,nrz2bnr8s6,1,2025-02-20 16:45:01,2025-02-21 12:47:44
8,862,1,r1cgy5kq5bviueq5,1,2025-02-20 16:42:31,2025-02-28 16:28:21
9,3251,2,uce50kz7so3,1,2025-02-20 16:51:15,2025-02-25 10:42:32
10,2364,1,uck1pr4ofirj2a513,1,2025-02-20 16:46:03,2025-02-28 12:50:36


**[예제 19]**  count 집계 함수를 사용해봅니다.

- count(*)는 결측을 포함한 카운팅, count(표현식)은 표현식의 결과값에 NULL을 제외하고 카운팅을 합니다.

In [78]:
%%SQL

SELECT count(*), count(rent_date), count(return_date) FROM rent1;

COUNT(*)              10
COUNT(RENT_DATE)       8
COUNT(RETURN_DATE)     5
Name: 1, dtype: int64

- count(DISTINCT 표현식)으로는 고유한 값의 카운팅을 할 수 있습니다.

reutrn_date에서 결측을 제외한 값을 가져왔습니다.
그리고 연월일(YYYYMMDD) 형식의 문자열로 변환합니다.

In [79]:
%%SQL

SELECT to_char(return_date, 'YYYYMMDD') FROM rent1 WHERE return_date is not NULL

,"TO_CHAR(RETURN_DATE,'YYYYMMDD')"
1,20250224
2,20250221
3,20250228
4,20250225
5,20250228


위 표현식에 DISTINCT를 앞에 붙인것의 차이를 살펴보겠습니다.

In [80]:
%%SQL

SELECT 
    count(DISTINCT to_char(return_date, 'yyyymmdd')), 
    count(to_char(return_date, 'yyyymmdd'))
FROM rent1 WHERE return_date is not NULL

COUNT(DISTINCTTO_CHAR(RETURN_DATE,'YYYYMMDD'))    4
COUNT(TO_CHAR(RETURN_DATE,'YYYYMMDD'))            5
Name: 1, dtype: int64

In [81]:
%%SQL

SELECT * FROM bookstat -- bookstat의 내용을 가져옵니다.

,book_id,stat_ymd,rent_count,rating_sum,rating_count,stat_date
1,142,20220929,1,24,3,2025-09-12 04:18:07
2,3405,20240308,1,29,3,2025-09-12 04:18:07
3,2222,20220408,1,28,3,2025-09-12 04:18:07
4,2046,20250126,1,27,3,2025-09-12 04:18:07
5,2184,20221029,1,25,3,2025-09-12 04:18:07
6,1895,20230703,1,25,3,2025-09-12 04:18:07
7,1607,20210723,1,27,3,2025-09-12 04:18:07
8,1111,20230625,1,26,3,2025-09-12 04:18:07
9,270,20230213,1,28,3,2025-09-12 04:18:07


- rating_count가 3을 초과한 bookstat을 가져옵니다.

In [82]:
%%SQL

SELECT * FROM bookstat WHERE rating_count > 3

,book_id,stat_ymd,rent_count,rating_sum,rating_count,stat_date


- rating_count가 3을 초과한 경우는 없는데요, 비어 있는 테이블의 카운트를 봅니다.

In [83]:
%%SQL

SELECT COUNT(*) FROM bookstat WHERE rating_count > 3

COUNT(*)    0
Name: 1, dtype: int64

※ **주의** 이런 헷갈림이 있는 경우가 시험에 잘 나옵니다.

count가 아닌 다른 집계함수는 비어있는 테이블에 대한 집계 결과는 비어있는 테이블을 주는 건 아니고,

NULL값을 줍니다.

In [84]:
%%SQL

SELECT MAX(rating_sum), MIN(rating_sum), AVG(rating_sum) FROM bookstat WHERE rating_count > 3

MAX(RATING_SUM)    NULL
MIN(RATING_SUM)    NULL
AVG(RATING_SUM)    NULL
Name: 1, dtype: object

**SELECT의 표현식에서 집계 함수는 GROUP BY에 의한 그룹별로 집계 연산을 수행합니다.**

**[예제 20]** GROUP BY 구문과 집계 함수로 구성한 간단한 SQL을 살펴 봅니다,

In [85]:
%%SQL

SELECT * FROM book1 -- book1의 내용을 봅니다.

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


- book1에서 publisher 별 행의 수를 출력합니다.

In [86]:
%%SQL 

SELECT publisher, count(*) FROM book1
GROUP BY publisher;

,publisher,COUNT(*)
1,풀빛,3
2,책읽는고양이,2
3,문학사상,1
4,군자출판사,3
5,꿈을담는틀,1


- book1에서 pub_date에서 연도를 추출하여, 연도별 행의 수를 출력합니다.

In [87]:
%%SQL 

SELECT extract(YEAR from pub_date) as year, count(*)
FROM book1
GROUP BY extract(YEAR from pub_date);

,year,COUNT(*)
1,2004,2
2,2000,1
3,2019,2
4,2022,1
5,2023,2
6,2024,2


- book1에서 pub_date에서 연도를 추출하고, 연도 별 publisher가 NULL이 아닌 수를 출력합니다.

In [88]:
%%SQL 

SELECT extract(YEAR from pub_date) as year, count(publisher)
FROM book1
GROUP BY extract(YEAR from pub_date);

,year,COUNT(PUBLISHER)
1,2004,2
2,2000,1
3,2019,2
4,2022,1
5,2023,2
6,2024,2


- book1에서 pub_date에서 연도를 추출하고, 연도 별 고유한 publisher의 수를 출력합니다.

In [89]:
%%SQL 

SELECT extract(YEAR from pub_date) as year, count(DISTINCT publisher)
FROM book1
GROUP BY extract(YEAR from pub_date);

,year,COUNT(DISTINCTPUBLISHER)
1,2004,2
2,2000,1
3,2019,1
4,2022,1
5,2023,1
6,2024,2


- book1에서 pub_date에서 연도를 추출하고, 연도와 publisher 별 행의 수를 출력합니다.

In [90]:
%%SQL

SELECT extract(YEAR from pub_date) as year, publisher, count(*)
FROM book1
GROUP BY extract(YEAR from pub_date), publisher;

,year,publisher,COUNT(*)
1,2004,풀빛,1
2,2004,책읽는고양이,1
3,2000,문학사상,1
4,2019,풀빛,2
5,2022,책읽는고양이,1
6,2023,군자출판사,2
7,2024,군자출판사,1
8,2024,꿈을담는틀,1


- NULL값에 집계 결과를 확인합니다.

In [91]:
%%SQL

SELECT * FROM rent1

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,2243,2,1jhmhlrlo71brwi4o8,1,NULL,NULL
2,970,1,46vpd6v91zb01e,1,2025-02-20 16:52:29,NULL
3,2403,2,4e1dh553s37,1,2025-02-20 16:43:29,2025-02-24 15:20:23
4,3194,3,94zfx2tf92heh59z,1,NULL,NULL
5,299,1,haejt93chfpl,1,2025-02-20 16:52:14,NULL
6,2239,1,ll65teqf4kzep8l1r,1,2025-02-20 16:56:31,NULL
7,2180,2,nrz2bnr8s6,1,2025-02-20 16:45:01,2025-02-21 12:47:44
8,862,1,r1cgy5kq5bviueq5,1,2025-02-20 16:42:31,2025-02-28 16:28:21
9,3251,2,uce50kz7so3,1,2025-02-20 16:51:15,2025-02-25 10:42:32
10,2364,1,uck1pr4ofirj2a513,1,2025-02-20 16:46:03,2025-02-28 12:50:36


In [92]:
%%SQL

SELECT to_char(return_date, 'yyyy/mm/dd'), count(*)
FROM rent1
GROUP BY to_char(return_date, 'yyyy/mm/dd')

,"TO_CHAR(RETURN_DATE,'YYYY/MM/DD')",COUNT(*)
1,NULL,5
2,2025/02/25,1
3,2025/02/28,2
4,2025/02/21,1
5,2025/02/24,1


**SELECT의 표현식들이 출력하는 행의 수는 동일해야 합니다.**

**[예제 21]** SELECT 표현식의 출력수의 불일치에 의한 오류의 예

In [93]:
%%SQL

SELECT substr(stat_ymd, 1, 4), rating_sum
FROM bookstat
GROUP BY substr(stat_ymd, 1, 4);

ORA-00979: not a GROUP BY expression None


|표현식|행의 수|
|----|----|
|substr(stat_ymd, 1, 4)|GROUP BY에 표현식에 포함되어 그룹당 1개|
|rating_sum| N개|

N: bookstat 테이블의 행의 수

In [94]:
%%SQL

SELECT substr(stat_ymd, 1, 4), sum(rating_sum)
FROM bookstat;

ORA-00937: not a single-group group function None


|표현식|행의 수|
|----|----|
|substr(stat_ymd, 1, 4)|N 개|
|sum(rating_sum)| 1개(그룹 1개)|

N: bookstat 테이블의 행의 수

**[예제 22]** GROUP BY와 복수의 집계 함수를 사용합니다.

In [95]:
%%SQL

SELECT * FROM bookstat -- bookstat의 내용을 봅니다.

,book_id,stat_ymd,rent_count,rating_sum,rating_count,stat_date
1,142,20220929,1,24,3,2025-09-12 04:18:07
2,3405,20240308,1,29,3,2025-09-12 04:18:07
3,2222,20220408,1,28,3,2025-09-12 04:18:07
4,2046,20250126,1,27,3,2025-09-12 04:18:07
5,2184,20221029,1,25,3,2025-09-12 04:18:07
6,1895,20230703,1,25,3,2025-09-12 04:18:07
7,1607,20210723,1,27,3,2025-09-12 04:18:07
8,1111,20230625,1,26,3,2025-09-12 04:18:07
9,270,20230213,1,28,3,2025-09-12 04:18:07


- bookstat에서 stat_ymd에서 1에서 4째자리까지의 문자를 취하고, 이를 기준으로 그룹별 합계와 최대값과 최소값을 구합니다.

In [96]:
%%SQL

SELECT substr(stat_ymd, 1, 4), sum(rating_sum), max(rating_sum), min(rating_sum)
FROM bookstat
GROUP BY substr(stat_ymd, 1, 4);

,"SUBSTR(STAT_YMD,1,4)",SUM(RATING_SUM),MAX(RATING_SUM),MIN(RATING_SUM)
1,2022,77,28,24
2,2024,29,29,29
3,2025,27,27,27
4,2023,79,28,25
5,2021,27,27,27


- 비어 있는 테이블의 GROUP BY 집계 결과는

In [97]:
%%SQL

SELECT * FROM bookstat WHERE rating_count > 3

,book_id,stat_ymd,rent_count,rating_sum,rating_count,stat_date


In [98]:
%%SQL

SELECT substr(stat_ymd, 1, 4), sum(rating_count) FROM bookstat
WHERE rating_count > 3
GROUP BY substr(stat_ymd, 1, 4);

,"SUBSTR(STAT_YMD,1,4)",SUM(RATING_COUNT)


**※ 비어 있는 테이블의 GROUP BY 집계 결과는 비어 있는 테이블입니다.**

- 복수의 그룹핑 기준에서도 각 열의 출력의 수는 일치해야 합니다.

In [99]:
%%SQL

SELECT extract(YEAR from pub_date) as year, publisher, pub_date
FROM book1
GROUP BY extract(YEAR from pub_date), publisher;

ORA-00979: not a GROUP BY expression None


|표현식|행의 수|
|----|----|
|extract(YEAR from pub_date) as year|GROUP BY에 표현식에 포함되어 그룹당 1개|
|publisher|GROUP BY에 표현식에 포함되어 그룹당 1개|
|pub_date|N개|

N: bookstat 테이블의 행의 수

**[예제 23]** 복수의 그룹핑 기준을 사용해 봅니다.

In [100]:
%%SQL

SELECT * FROM book1

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


- pub_date에서 연도와 publisher로 그룹핑하여 pub_date의 최대값을 출력합니다.

In [101]:
%%SQL

SELECT extract(YEAR from pub_date) as year, publisher, max(pub_date)
FROM book1
GROUP BY extract(YEAR from pub_date), publisher;

,year,publisher,MAX(PUB_DATE)
1,2004,풀빛,2004-01-01
2,2004,책읽는고양이,2004-07-01
3,2000,문학사상,2000-10-01
4,2019,풀빛,2019-10-01
5,2022,책읽는고양이,2022-07-01
6,2023,군자출판사,2023-05-01
7,2024,군자출판사,2024-11-01
8,2024,꿈을담는틀,2024-10-01


|표현식|행의 수|
|----|----|
|extract(YEAR from pub_date) as year|GROUP BY에 표현식에 포함되어 그룹당 1개|
|publisher|GROUP BY에 표현식에 포함되어 그룹당 1개|
|max(pub_date)|GROUP BY에 표현식에 포함되어 그룹당 1개|

N: bookstat 테이블의 행의 수

## HAVING

GROUP BY에 대한 집계 연산의 결과들로 조건을 만들어 행들을 선택


**[예제 24]** HAVING을 사용하여 GROUP BY 연산 결과를 이용한 선택을 합니다.

In [102]:
%%SQL

SELECT publisher, count(*)
FROM book1
GROUP BY publisher;

,publisher,COUNT(*)
1,풀빛,3
2,책읽는고양이,2
3,문학사상,1
4,군자출판사,3
5,꿈을담는틀,1


- book1의 publisher 중 행의 수가 1개를 초과한 publisher를 가져옵니다.

In [103]:
%%SQL
SELECT publisher, count(*) cnt
FROM book1
WHERE count(*) > 1
GROUP BY publisher;

ORA-00934: group function is not allowed here None


In [104]:
%%SQL

SELECT publisher
FROM book1
GROUP BY publisher
HAVING count(*) > 1;

,publisher
1,풀빛
2,책읽는고양이
3,군자출판사


In [105]:
%%SQL
SELECT publisher
FROM book1
WHERE book_categ_id < 1000
GROUP BY publisher
HAVING count(*) > 1;

,publisher
1,책읽는고양이
2,풀빛
3,군자출판사


- GROUP BY 표현식을 HAVING에서 사용할 수 있습니다.

In [106]:
%%SQL

SELECT publisher
FROM book1
GROUP BY publisher
HAVING publisher like '%사';

publisher    군자출판사
Name: 1, dtype: object

- GROUP BY 표현식에 없거나, 출력행의 수가 다를 경우에는 쿼리가 성립하지 않습니다.

In [107]:
%%SQL

SELECT publisher
FROM book1
GROUP BY publisher
HAVING extract(YEAR from pub_date) >= 2000;

ORA-00979: not a GROUP BY expression None


- 하지만 GROUPBY 보다 연산 순위가 앞선 WHERE에서는 GROUPBY와 행의 개수가 달라도 상관없습니다.

In [108]:
%%SQL

SELECT publisher
FROM book1
WHERE extract(YEAR from pub_date) >= 2000
GROUP BY publisher;

,publisher
1,풀빛
2,책읽는고양이
3,문학사상
4,군자출판사
5,꿈을담는틀


## GROUP 함수

**GROUP BY 절에서 사용하는 함수**

|함수|기능|
|----|-----|
|ROLLUP|ROLLUP은 GROUP BY의 계층을 하나씩 높여 가면서 소계 연산을 수행|
|CUBE|GROUP BY 컬럼의 모든 조합에 대한 소계 연산을 수행|
|GROUPING SETS|GROUP BY 각 컬럼별 그룹핑 연산을 수행|


**[예제 25]** GROUP 함수를 사용하여 데이터 추출을 해봅니다.

In [109]:
%%SQL

SELECT count(*) FROM rating -- rating은 데이터의 수가 약 23000개 입니다.

COUNT(*)    103404
Name: 1, dtype: int64

In [110]:
%%SQL

SELECT * FROM rating -- rating으 내용을 봅니다.

,book_id,member_id,rating,rate_date
1,6,g9h5abtvl4py6c3z4,10,2021-12-08 13:39:44
2,6,go5lwnkx36ay,9,2020-10-11 15:30:12
3,6,grzfpmuao1b0,8,2020-07-16 14:34:42
4,6,htkhgeto9kdh,8,2024-03-04 12:26:58
5,6,iizxw855kaut3,9,2021-09-21 09:23:51
...,...,...,...,...
103400,3033,vo3x1h9uptzqj,7,2022-12-01 15:50:56
103401,3033,vrk43uoz610a7,10,2020-12-30 16:48:52
103402,3033,ww5qd8b0bnfz,10,2021-10-01 10:23:34
103403,3033,x3ionc0swm,8,2019-10-11 10:50:25


- GROUP 함수를 사용하지 않은 GROUP BY 구문입니다. rating의 rate_date에서 월과 요일을 추출하고, 이들을 그룹핑의 기준으로, rating의 합과, 행의 개수를 출력하는 SQL을 작성합니다.

In [111]:
%%SQL

SELECT extract(MONTH from rate_date), to_char(rate_date, 'DY'), 
    sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY extract(MONTH from rate_date), to_char(rate_date, 'DY');

,EXTRACT(MONTHFROMRATE_DATE),"TO_CHAR(RATE_DATE,'DY')",SUM(RATING),COUNT(*)
1,2,WED,1391,160
2,1,WED,2749,318
3,1,THU,2698,310
4,1,SUN,2202,255
5,2,SUN,1428,166
6,1,FRI,2857,330
7,1,SAT,2184,250
8,1,MON,1948,224
9,2,MON,1171,133
10,1,TUE,2008,235


- 위 구문에서 GROUP BY의 ROLLUP을 사용합니다.

In [112]:
%%SQL

SELECT extract(MONTH from rate_date), to_char(rate_date, 'DY'), 
    sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY ROLLUP(extract(MONTH from rate_date), to_char(rate_date, 'DY'));

,EXTRACT(MONTHFROMRATE_DATE),"TO_CHAR(RATE_DATE,'DY')",SUM(RATING),COUNT(*)
1,2.0,WED,1391,160
2,1.0,WED,2749,318
3,1.0,THU,2698,310
4,1.0,SUN,2202,255
5,2.0,SUN,1428,166
6,1.0,FRI,2857,330
7,1.0,SAT,2184,250
8,1.0,MON,1948,224
9,2.0,MON,1171,133
10,1.0,TUE,2008,235


※ ROLLUP은 GROUP BY의 계층을 하나씩 높여 가면서 소계 작업까지를 수행합니다.

- 기본 GROUP BY SQL에 book_id 기준을 가장 앞에 추가하고 GROUP 함수 CUBE를 사용합니다.

In [113]:
%%SQL
SELECT book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY'), 
    sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY');

,book_id,EXTRACT(MONTHFROMRATE_DATE),"TO_CHAR(RATE_DATE,'DY')",SUM(RATING),COUNT(*)
1,2,2,SUN,8,1
2,2,1,THU,8,1
3,5,2,THU,10,1
4,42,1,WED,9,1
5,47,2,WED,10,1
...,...,...,...,...,...
2641,3193,2,TUE,10,1
2642,3279,1,MON,8,1
2643,3279,1,WED,9,1
2644,3015,2,THU,10,1


In [114]:
%%SQL

SELECT book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY'), 
    sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY CUBE(book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY'));

,book_id,EXTRACT(MONTHFROMRATE_DATE),"TO_CHAR(RATE_DATE,'DY')",SUM(RATING),COUNT(*)
1,NULL,NULL,NULL,26089,3010
2,NULL,NULL,FRI,3851,442
3,NULL,NULL,MON,3119,357
4,NULL,NULL,SAT,3744,435
5,NULL,NULL,SUN,3630,421
...,...,...,...,...,...
7570,3563.0,1.0,WED,10,1
7571,3564.0,NULL,NULL,8,1
7572,3564.0,NULL,THU,8,1
7573,3564.0,2.0,NULL,8,1


※ CUBE는 GROUP BY의 뎁스별로 조합가능한 모든 그룹기준으로 소계 작업까지  수행합니다.

- 기본의 GROUP BY SQL에 book_id 기준을 가장 앞에 추가하고 GROUP 함수 GROUPING SET를 사용합니다.

In [115]:
%%SQL

SELECT book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY'), 
    sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY GROUPING SETS(book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY'));

,book_id,EXTRACT(MONTHFROMRATE_DATE),"TO_CHAR(RATE_DATE,'DY')",SUM(RATING),COUNT(*)
1,NULL,NULL,WED,4140,478
2,NULL,NULL,THU,4435,505
3,NULL,NULL,SUN,3630,421
4,NULL,NULL,FRI,3851,442
5,NULL,NULL,SAT,3744,435
...,...,...,...,...,...
1030,3194.0,NULL,NULL,53,6
1031,2997.0,NULL,NULL,19,2
1032,3198.0,NULL,NULL,55,6
1033,3280.0,NULL,NULL,8,1


- GROUPING SETS는 항목별 GROUP BY를 하여 각각의 GROUP BY 결과를 결합하여 보여줍니다.

In [116]:
%%SQL

SELECT book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY'), 
    sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY GROUPING SETS(
    book_id, 
    (book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY')), 
    to_char(rate_date, 'DY'),
    (extract(MONTH from rate_date), to_char(rate_date, 'DY'))
);

,book_id,EXTRACT(MONTHFROMRATE_DATE),"TO_CHAR(RATE_DATE,'DY')",SUM(RATING),COUNT(*)
1,11.0,1.0,SAT,7,1
2,16.0,1.0,MON,10,1
3,40.0,1.0,WED,8,1
4,42.0,2.0,THU,10,1
5,44.0,1.0,FRI,9,1
...,...,...,...,...,...
3687,NULL,NULL,WED,4140,478
3688,NULL,NULL,THU,4435,505
3689,NULL,NULL,FRI,3851,442
3690,NULL,NULL,SUN,3630,421


In [117]:
%%SQL

SELECT * FROM (
    SELECT book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY'), 
        sum(rating), count(*)
    FROM rating
    WHERE extract(YEAR from rate_date) = 2025
    GROUP BY GROUPING SETS(book_id, extract(MONTH from rate_date), to_char(rate_date, 'DY'))
) WHERE book_id is NULL
;

,book_id,EXTRACT(MONTHFROMRATE_DATE),"TO_CHAR(RATE_DATE,'DY')",SUM(RATING),COUNT(*)
1,NULL,NULL,WED,4140,478
2,NULL,NULL,THU,4435,505
3,NULL,NULL,SUN,3630,421
4,NULL,NULL,FRI,3851,442
5,NULL,NULL,SAT,3744,435
6,NULL,NULL,MON,3119,357
7,NULL,NULL,TUE,3170,372
8,NULL,2.0,NULL,9443,1088
9,NULL,1.0,NULL,16646,1922


※ GROUPING SETES는 GROUP BY의 표현식 별로 소계 연산은 하지 않고,  GROUP BY에서 다루지 않았던 것이 NULL로 들어갑니다.

### GROUPING 지원함수
    
|함수|기능|
|----|----|
|GROUPING|각 필드의 값이 소계 연산에 의한 것인지 파악<br/>CASE 구문과 조합하여 소계 연산에 대한 결과값을 지정<br/> **0**: 일반 groupby 연산, **1**: 소계 연산|


**[예제 26]** GROUPING 지원 함수를 활용해봅니다.

- GROUPING 연선 결과를 확인합니다.

In [118]:
%%SQL

SELECT extract(MONTH from rate_date), GROUPING(extract(MONTH from rate_date)), to_char(rate_date, 'DY'), GROUPING(to_char(rate_date, 'DY')),
    sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY GROUPING SETS(extract(MONTH from rate_date), to_char(rate_date, 'DY'));

,EXTRACT(MONTHFROMRATE_DATE),GROUPING(EXTRACT(MONTHFROMRATE_DATE)),"TO_CHAR(RATE_DATE,'DY')","GROUPING(TO_CHAR(RATE_DATE,'DY'))",SUM(RATING),COUNT(*)
1,2.0,0,NULL,1,9443,1088
2,1.0,0,NULL,1,16646,1922
3,NULL,1,WED,0,4140,478
4,NULL,1,THU,0,4435,505
5,NULL,1,SUN,0,3630,421
6,NULL,1,FRI,0,3851,442
7,NULL,1,SAT,0,3744,435
8,NULL,1,MON,0,3119,357
9,NULL,1,TUE,0,3170,372


- GROUPING을 이용하여 소계 연산 결과의 이름을 붙입니다.

In [119]:
%%SQL

SELECT CASE 
        WHEN GROUPING(extract(MONTH from rate_date)) = 0 THEN to_char(extract(MONTH from rate_date)) 
        ELSE '총합' 
    END as 월, 
    CASE 
        WHEN GROUPING(to_char(rate_date, 'DY')) = 0  THEN to_char(rate_date, 'DY')
        ELSE '소계'
    END as 요일, sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY ROLLUP(extract(MONTH from rate_date), to_char(rate_date, 'DY'));

,월,요일,SUM(RATING),COUNT(*)
1,2,WED,1391,160
2,1,WED,2749,318
3,1,THU,2698,310
4,1,SUN,2202,255
5,2,SUN,1428,166
6,1,FRI,2857,330
7,1,SAT,2184,250
8,1,MON,1948,224
9,2,MON,1171,133
10,1,TUE,2008,235


# ORDER BY 절

```SQL
SELECT [DISTINCT] 표현식1 [as 명칭1], 표현식2 [as 명칭2], …
FROM 테이블명 [ALIAS]
[WHERE 조건식]
[GROUP BY 표현식1[, 표현식2, …]
[HAVING 그룹조건식]
[ORDER BY {컬럼 순서1|표현식1} [{ASC| DESC}][, {컬럼 순서2|표현식2}[{ASC|DESC}] …];
```

- ASC(Ascending): 오름차순, DESC(Descending): 내림차순
- 기본 정렬 순서는 ASC 오름차순입니다.
- 먼저 정의된 표현식의 출력값이 우선 기준이 됩니다.

**[예제 27]** ORDER BY를 통해 정렬 연산을 수행합니다.

- book1에서 첫 번째 기준으로 publisher는 오름차수, pub_date 내림 차순으로 정렬합니다.

In [120]:
%%SQL

SELECT * 
FROM book1 
ORDER BY publisher, pub_date DESC 

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
2,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
3,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
4,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54
5,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
8,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
9,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
10,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07


- ORDER BY 에서는 SELECT에서 선택하지 않은 컬럼도 사용 가능 합니다.

In [121]:
%%SQL

SELECT pub_date
FROM book1 
ORDER BY publisher, pub_date DESC 

,pub_date
1,2024-11-01
2,2023-05-01
3,2023-02-01
4,2024-10-01
5,2000-10-01
6,2022-07-01
7,2004-07-01
8,2019-10-01
9,2019-07-01
10,2004-01-01


- GROUP BY가 사용될 경우 GROUP BY 집계 함수도 사용 가능

In [122]:
%%SQL

SELECT CASE 
        WHEN GROUPING(extract(MONTH from rate_date)) = 0 THEN to_char(extract(MONTH from rate_date)) 
        ELSE '총합' 
    END as 월, 
    CASE 
        WHEN GROUPING(to_char(rate_date, 'DY')) = 0  THEN to_char(rate_date, 'DY')
        ELSE DECODE(GROUPING(extract(MONTH from rate_date)), 0, '소계', '-')
    END as 요일, sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY ROLLUP(extract(MONTH from rate_date), to_char(rate_date, 'DY'));
ORDER BY 1, 2

,월,요일,SUM(RATING),COUNT(*)
1,1,FRI,2857,330
2,1,MON,1948,224
3,1,SAT,2184,250
4,1,SUN,2202,255
5,1,THU,2698,310
6,1,TUE,2008,235
7,1,WED,2749,318
8,1,소계,16646,1922
9,2,FRI,994,112
10,2,MON,1171,133


In [123]:
%%SQL

SELECT CASE 
        WHEN GROUPING(extract(MONTH from rate_date)) = 0 THEN to_char(extract(MONTH from rate_date)) 
        ELSE '총합' 
    END as 월, 
    CASE 
        WHEN GROUPING(to_char(rate_date, 'DY')) = 0  THEN to_char(rate_date, 'DY')
        ELSE DECODE(GROUPING(extract(MONTH from rate_date)), 0, '소계', '-')
    END as 요일, sum(rating), count(*)
FROM rating
WHERE extract(YEAR from rate_date) = 2025
GROUP BY ROLLUP(extract(MONTH from rate_date), to_char(rate_date, 'DY'));
ORDER BY 1, SUM(RATING)

,월,요일,SUM(RATING),COUNT(*)
1,1,MON,1948,224
2,1,TUE,2008,235
3,1,SAT,2184,250
4,1,SUN,2202,255
5,1,THU,2698,310
6,1,WED,2749,318
7,1,FRI,2857,330
8,1,소계,16646,1922
9,2,FRI,994,112
10,2,TUE,1162,137


- Oracle에서는 NULL 값은 가장 큰 값으로 간주 / SQL Server는 NULL 값은 가장 작은 값으로 간주

In [124]:
%%SQL

SELECT * 
FROM rent1
ORDER BY rent_date

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,862,1,r1cgy5kq5bviueq5,1,2025-02-20 16:42:31,2025-02-28 16:28:21
2,2403,2,4e1dh553s37,1,2025-02-20 16:43:29,2025-02-24 15:20:23
3,2180,2,nrz2bnr8s6,1,2025-02-20 16:45:01,2025-02-21 12:47:44
4,2364,1,uck1pr4ofirj2a513,1,2025-02-20 16:46:03,2025-02-28 12:50:36
5,3251,2,uce50kz7so3,1,2025-02-20 16:51:15,2025-02-25 10:42:32
6,299,1,haejt93chfpl,1,2025-02-20 16:52:14,NULL
7,970,1,46vpd6v91zb01e,1,2025-02-20 16:52:29,NULL
8,2239,1,ll65teqf4kzep8l1r,1,2025-02-20 16:56:31,NULL
9,2243,2,1jhmhlrlo71brwi4o8,1,NULL,NULL
10,3194,3,94zfx2tf92heh59z,1,NULL,NULL


# SELECT 문장의 실행 순서

**테이블 설정**

1. 대상 테이블 설정(FROM) \[→ 2. 대상 행 한정(WHERE)\] \[→ 3. 행 그룹핑(GROUP BY) \[4. 그룹핑한 행들의 집결 결과에서 한정(HAVING)\] \]

**컬럼 선택**

2. 설정된 테이블에서 조회할 컬럼을 선택(SELECT)

**행 정렬**

3. 선택된 컬럼을 기준으로 (ORDER BY)



```SQL
5. SELECT [DISTINCT] 표현식1 [as 명칭1], 표현식2 [as 명칭2], …
1. FROM 테이블명 [ALIAS]
2. [WHERE 조건식]
3. [GROUP BY 표현식1[, 표현식2, …]
4. [HAVING 그룹조건식]
6. [ORDER BY {컬럼 순서1|표현식1} [{ASC| DESC}][, {컬럼 순서2|표현식2}[{ASC|DESC}, …];
```

**[예제 28]** 예제 SQL을 통해 SELECT 문장의 실행 순서를 살펴 봅니다.

In [125]:
%%SQL

SELECT to_char(rate_date, 'yyyy-mm-dd') as 일자, 
        avg(rating) as 평균평점, count(rating) as 평점수    -- 5. FROM ~ HAVING 까지 설정된 테이블에서 컬럼 선택
FROM rating                                                 -- 1. FROM을 통해 대상 테이블 지장
WHERE extract(YEAR from rate_date) in (2023, 2024)          -- 2. 테이블에서 행 한정
GROUP BY to_char(rate_date, 'yyyy-mm-dd')                   -- 3. 한정된 행에서 그룹핑
HAVING count(RATING) >= 5                                   -- 4. 그룹핑한 행들의 집계 결과에서 한정
ORDER BY 2                                                  -- 6. 조회한 데이터에서 정렬 (SELECT의 출력 순서를 숫자로 지정)

,일자,평균평점,평점수
1,2023-04-04,8,57
2,2023-07-31,8.03921568627450980392156862745098039216,51
3,2023-05-31,8.0952380952380952380952380952380952381,42
4,2023-04-19,8.10416666666666666666666666666666666667,48
5,2023-02-05,8.125,56
...,...,...,...
727,2023-12-10,9.11111111111111111111111111111111111111,45
728,2023-12-11,9.125,40
729,2024-07-09,9.14,50
730,2024-05-12,9.14285714285714285714285714285714285714,28


In [126]:
%%SQL

SELECT to_char(rate_date, 'yyyy-mm-dd') as 일자, 
        avg(rating) as 평균평점, count(rating) as 평점수    -- 5. FROM ~ HAVING 까지 설정된 테이블에서 컬럼 선택
FROM rating                                                 -- 1. FROM을 통해 대상 테이블 지장
WHERE 일자 in (2023, 2024)                                  -- 2. 테이블에서 행 한정 (나중에 수행되는 SELECT의 내용은 WHERE에서 참조하지 못합니다.)
GROUP BY to_char(rate_date, 'yyyy-mm-dd')                   -- 3. 한정된 행에서 그룹핑
HAVING count(RATING) >= 5                                   -- 4. 그룹핑한 행들의 집계 결과에서 한정
ORDER BY 2                                                  -- 6. 조회한 데이터에서 정렬

ORA-00904: "일자": invalid identifier None


In [127]:
%%SQL

SELECT to_char(rate_date, 'yyyy-mm-dd') as 일자, 
        avg(rating) as 평균평점, count(rating) as 평점수    -- 5. FROM ~ HAVING 까지 설정된 테이블에서 컬럼 선택
FROM rating                                                 -- 1. FROM을 통해 대상 테이블 지장
WHERE extract(YEAR from rate_date) in (2023, 2024)          -- 2. 테이블에서 행 한정
GROUP BY to_char(rate_date, 'yyyy-mm-dd')                   -- 3. 한정된 행에서 그룹핑
HAVING count(RATING) >= 5                                   -- 4. 그룹핑한 행들의 집계 결과에서 한정
ORDER BY 평점수 DESC                                        -- 6. 조회한 데이터에서 정렬 (ORDER BY에서는 SELECT 구문보다 이후에 수행되므로 참조가 가능하죠) 

,일자,평균평점,평점수
1,2023-11-04,8.90588235294117647058823529411764705882,85
2,2024-09-12,8.86904761904761904761904761904761904762,84
3,2024-10-24,8.5180722891566265060240963855421686747,83
4,2024-01-19,8.59756097560975609756097560975609756098,82
5,2024-09-14,8.70731707317073170731707317073170731707,82
...,...,...,...
727,2024-02-18,8.27272727272727272727272727272727272727,22
728,2023-01-28,8.7,20
729,2024-02-17,9,18
730,2024-02-15,8.88235294117647058823529411764705882353,17


In [128]:
%%SQL

SELECT to_char(rate_date, 'yyyy-mm-dd') as 일자, 
        avg(rating) as 평균평점, count(rating) as 평점수   -- 5. FROM ~ HAVING 까지 설정된 테이블에서 컬럼 선택
FROM rating                                                -- 1. FROM을 통해 대상 테이블 지장
WHERE extract(YEAR from rate_date) in (2023, 2024)         -- 2. 테이블에서 행 한정
GROUP BY 일자                                              -- 3. 에러: 순서항 SELECT 보다 앞서므로, SELECT에서 지정된 '일자' 컬럼을 사용할 수 없습니다. 
HAVING count(RATING) >= 5                                  -- 4. 그룹핑한 행들의 집계 결과에서 한정
ORDER BY 평점수 DESC                                       -- 6. 조회한 데이터에서 정렬

ORA-00904: "일자": invalid identifier None


# JOIN

## Join의 개요

두 개 이상의 테이블을 결합

기본적으로 PK(Primary Key), FK(Foreign Key)와의 관계를 기반으로 수행

PK와 FK 관계가 없이도 논리적인 값들의 연산을 통해서도 수행 가능

|조인|설명|용법|
|------|-----|------|
|EQUI JOIN|ON 에서 결합 조건에 해당하는 행들만을 출력|SELECT 테이블1.컬럼명, 테이블2.컬럼명, … FROM<br/> 테이블1, 테이블2<br/>WHERE 테이블1.컬럼명 = 테이블2.컬럼명 [AND 테이블1.컬럼명2 = 테이블2.컬럼명2)|
|Non-EQUI JOIN| 두 테이블의 모든 동일한 컬럼명의 값이 같을 경우만 출력<br/>USING은 동일한 컬럼명중 비교에 대상을 한정할 경우 사용|SELECT 테이블1.컬럼명, 테이블2.컬럼명, …<br/>FROM 테이블1, 테이블2<br/>\[WHERE 임의의 조건\]<br/><br/>만일 결합 조건이 없다면, 테이블1, 테이블2의 교차 조합이 반환된다.|
|3개 이상의 테이블|3개 이상의 테이블을 조인|SELECT 테이블1.컬럼명, 테이블2.컬럼명, 테이블3.컬럼명,…<br/>FROM 테이블1, 테이블2, 테이블3, …<br/>[WHERE 임의의 조건]|
|OUTER JOIN|테이블1의 행들은 모두 포함하고 ON을 만족하는 테이블2의 행들을 결합.<br/>테이블2에서의 결합 데이터가 없을 경우 NULL값|SELECT 테이블1.컬럼명, 테이블2.컬럼명<br/>FROM 테이블1, 테이블2<br/>WHERE {테이블1.컬럼명 = 테이블2.컬럼명(+)\[AND 테이블1.컬럼명2 = 테이블.컬럼명2(+) …\] | <br/>     테이블1.컬럼명(+) = 테이블2.컬럼명 \[AND 테이블1.컬럼명2(+) = 테이블.컬럼명2\]<br/><br/>컬럼에 (+)가 붙은 테이블이 기준외 테이블에 해당|

※ Oracle DBMS는 원형 수준의 Join 연산을 최적화 처리기를 통해 구현이 되었습니다.

**[예제 29]** Join의 기본 개념

- rating1에서 평점수와 평균 평점을 구하고, member1과 EQUI Join 결합을 합니다.

In [129]:
%%SQL
SELECT * FROM member1

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길


In [130]:
%%SQL
SELECT member_id, count(*) as "평점수", avg(rating) as "평균 평점" FROM rating
GROUP BY member_id

,member_id,평점수,평균 평점
1,irj1h2cdtfs1r6png,3,8.66666666666666666666666666666666666667
2,nogrbie1ad26r,3,9
3,ny6xbm6l9r3,6,8.66666666666666666666666666666666666667
4,wcvorqtub5v,11,9.09090909090909090909090909090909090909
5,1ntj3clnmz,9,9.22222222222222222222222222222222222222
...,...,...,...
19869,m0usscx1o9s,1,3
19870,itmncvfl8rtxm2,1,9
19871,ljzjoi2f3dv6dk,1,8
19872,tsth7i7t93ku,1,9


In [131]:
%%SQL

SELECT * FROM 
member1 a, (
    SELECT member_id, count(*) as "평점수", avg(rating) as "평균 평점" FROM rating
    GROUP BY member_id
) b
WHERE a.member_id = b.member_id

,member_id,member_name,region,address1,address2,member_id,평점수,평균 평점
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,zcitby9bvw,10,8.6
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길,9005r5vmi,7,9.14285714285714285714285714285714285714
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길,nnvy9vtclgyfb,6,9
4,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길,osfi8te4nv9m,5,7.2
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길,q6bl2tdplxz,3,9.66666666666666666666666666666666666667
6,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길,kijtmcsxqvxe4d,1,7
7,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,71r05w32wx4e5j,1,9


- bookstock에서 boo_id별 평균 입고 가격을 구하고 평균 입고 가격보다 큰 가격으로 구입한 거래를 Non-EQUI Join을 활용하여 조사합니다.

In [132]:
%%SQL

DESC bookstock

,column_name,data_type,data_length,nullable
1,BOOK_ID,NUMBER,22,N
2,STOCK_SEQ,NUMBER,22,N
3,STOCK_PRICE,NUMBER,22,N
4,BOOK_STATUS,NUMBER,22,N
5,LOCATION,VARCHAR2,50,Y
6,STOCK_DATE,DATE,7,Y
7,STATUS_DATE,DATE,7,Y


In [133]:
%%SQL

SELECT book_id, avg(stock_price) FROM bookstock
GROUP BY book_id

,book_id,AVG(STOCK_PRICE)
1,210,7916.666666666666666666666666666666666667
2,232,11305
3,237,12600
4,277,20000
5,281,14000
...,...,...
2627,3423,13770
2628,3440,12960
2629,3447,8100
2630,3452,14400


In [134]:
%%SQL

SELECT a.book_id, a.stock_seq, a.stock_date, a.stock_price, b.avg_stock_price FROM 
bookstock a , (
    SELECT book_id, avg(stock_price) avg_stock_price FROM bookstock
    GROUP BY book_id
) b
WHERE a.book_id = b.book_id and a.stock_price > avg_stock_price

,book_id,stock_seq,stock_date,stock_price,avg_stock_price
1,208,1,2022-12-28 15:19:05,20250,19873.3333333333333333333333333333333333
2,208,3,2023-02-09 15:06:18,20250,19873.3333333333333333333333333333333333
3,209,2,2021-08-16 11:51:34,4500,4160
4,210,1,2023-12-16 13:23:36,8190,7916.666666666666666666666666666666666667
5,210,2,2023-12-29 13:15:45,8190,7916.666666666666666666666666666666666667
...,...,...,...,...,...
1543,3472,2,2021-11-06 09:28:02,9180,8910
1544,3473,1,2024-12-24 09:45:58,10800,9720
1545,3474,1,2023-01-19 12:51:13,12560,11860
1546,3476,1,2020-11-09 12:23:53,11700,10820


- book1과 rent 테이블에서 book_id 별 대출건수, rating에서 평균 평점을 구해 book_id, book_title, rent_count, rating_avg를 출력합니다.

In [135]:
%%SQL
SELECT * FROM book1

,book_id,book_title,book_subtitle,book_categ_id,publisher,authors,summary,pub_date,reg_date
1,2459,수상한 과학,NULL,1185,풀빛,전방욱,생명공학 과학자들은 생명공학 기술이 가져다줄 무한한 이익을 보장하며 우리에게 장미빛...,2004-01-01,2004-01-01 15:58:07
2,1706,큰글씨 나는 이렇게 나이들고 싶다,소노 아야코가 마흔에 쓴 늙음을 경계하는 글,834,책읽는고양이,소노 아야코|오경순,나이듦이 처음인 우리 모두가 찾던 깊고도 명료한 메시지법정 스님이 《아름다운 마무리...,2004-07-01,2004-07-01 16:13:00
3,1065,상실의 시대,NULL,489,문학사상,무라카미 하루키|유유정,"오늘을 사는 젊은 세대들의 방황, 그들이 느끼는 상실감을 감동적으로 그려낸 무라카미...",2000-10-01,2000-10-01 10:03:21
4,1544,"궁금해요, 세종 대왕",NULL,731,풀빛,안선모|한용욱,백성을 자기 자신보다 더 사랑한 위대한 왕백성에 대한 세종의 사랑이 만들어 낸 가장...,2019-07-01,2019-07-01 09:50:31
5,971,뉴스를 보는 눈,가짜 뉴스를 선별하는 미디어 리터러시,448,풀빛,구본권,"지털 시민 역량을 키우는 핵심,미디어 리터러시비뚤어진 언론을 바로잡고 가짜 뉴스를 ...",2019-10-01,2019-10-01 16:27:43
6,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,898,책읽는고양이,황윤,색다른 스토리텔링으로 만나는 국립중앙박물관그 주인공은 금동반가사유상이 책은 국립중앙...,2022-07-01,2022-07-01 10:34:15
7,14,차근차근 배우는 우리 아이 일상 말하기,스크립트 중재 기반,7,군자출판사,홍경훈|윤미선|이수향|유지원|김하은|정보 더 보기/감추기|홍경훈|윤미선|이수향|유지...,언어치료 전문가가 알려주는 우리 아이의 일상생활 말하기 연습!『차근차근 배우는 우리...,2023-02-01,2023-02-01 11:10:01
8,1384,2024 SIMPLE 의료법규,NULL,654,군자출판사,길벗,“법의 무지는 용서되지 않는다(Ignorantia juris non excusat)...,2023-05-01,2023-05-01 12:38:16
9,704,임신부·모유수유부의 안전한 약물 사용,모체뿐만 아니라 아이의 건강까지 고려한 안전한 약물 정보 가이드,324,군자출판사,한국마더세이프,『임신부·모유수유부의 안전한 약물 사용』은 고령 가임 여성의 증가와 만성질환으로 약...,2024-11-01,2024-11-01 13:33:39
10,3327,중학 국어 기초 완성,NULL,1579,꿈을담는틀,EBS,꼭 알아야 할 중학 국어 지식 총정리!중학생이 꼭 알아야 할 필수 개념만 모아 한 ...,2024-10-01,2024-10-01 12:06:54


In [136]:
%%SQL
SELECT book_id, count(*) as rent_count FROM rent
GROUP BY book_id

,book_id,rent_count
1,1105,74
2,2708,55
3,1741,378
4,1578,861
5,124,185
...,...,...
1845,3518,1
1846,3527,1
1847,1502,2
1848,2997,28


In [137]:
%%SQL
SELECT book_id, avg(rating) as rating_avg FROM rating
GROUP BY booK_id

,book_id,rating_avg
1,6,8.82716049382716049382716049382716049383
2,14,8.32
3,23,8.78964401294498381877022653721682847896
4,27,9.2
5,50,9.625
...,...,...
1681,3280,8.80459770114942528735632183908045977011
1682,2808,1
1683,2819,8.88636363636363636363636363636363636364
1684,3021,1.33333333333333333333333333333333333333


In [138]:
%%SQL
SELECT a.book_id, a.book_title, b.rent_count, c.rating_avg FROM 
book1 a, (
    SELECT book_id, count(*) as rent_count FROM rent
    GROUP BY book_id
) b, (
    SELECT book_id, avg(rating) as rating_avg FROM rating
    GROUP BY booK_id
) c
WHERE a.book_id = b.book_id AND a.book_id = c.book_id

,book_id,book_title,rent_count,rating_avg
1,14,차근차근 배우는 우리 아이 일상 말하기,97,8.32
2,1706,큰글씨 나는 이렇게 나이들고 싶다,646,9.25165562913907284768211920529801324503
3,1544,"궁금해요, 세종 대왕",692,9.1215469613259668508287292817679558011
4,1065,상실의 시대,902,7.87681159420289855072463768115942028986
5,971,뉴스를 보는 눈,166,9.02040816326530612244897959183673469388
6,3327,중학 국어 기초 완성,24,9.4


- book1과 rent 테이블에서 book_id 별 대출건수, rating에서 평균 평점을 구해 book_id, book_title, rent_count, rating_avg를 출력합니다.

  여기서 대출이력이나 rating_avg 이력이 없으면 0을 출력합니다.

In [139]:
%%SQL
SELECT a.book_id, a.book_title, b.rent_count, c.rating_avg FROM 
book1 a, (
    SELECT book_id, count(*) as rent_count FROM rent
    GROUP BY book_id
) b, (
    SELECT book_id, avg(rating) as rating_avg FROM rating
    GROUP BY booK_id
) c
WHERE a.book_id = b.book_id(+) AND a.book_id = c.book_id(+)

,book_id,book_title,rent_count,rating_avg
1,14,차근차근 배우는 우리 아이 일상 말하기,97.0,8.32
2,1706,큰글씨 나는 이렇게 나이들고 싶다,646.0,9.25165562913907284768211920529801324503
3,1544,"궁금해요, 세종 대왕",692.0,9.1215469613259668508287292817679558011
4,1065,상실의 시대,902.0,7.87681159420289855072463768115942028986
5,971,뉴스를 보는 눈,166.0,9.02040816326530612244897959183673469388
6,3327,중학 국어 기초 완성,24.0,9.4
7,1873,"일상이 고고학, 나 혼자 국립중앙박물관",NULL,NULL
8,2459,수상한 과학,NULL,NULL
9,704,임신부·모유수유부의 안전한 약물 사용,19.0,NULL
10,1384,2024 SIMPLE 의료법규,NULL,NULL


In [140]:
%%SQL
SELECT a.book_id, a.book_title, nvl(b.rent_count, 0), nvl(c.rating_avg, 0) FROM 
book1 a, (
    SELECT book_id, count(*) as rent_count FROM rent
    GROUP BY book_id
) b, (
    SELECT book_id, avg(rating) as rating_avg FROM rating
    GROUP BY booK_id
) c
WHERE a.book_id = b.book_id(+) AND a.book_id = c.book_id(+)

,book_id,book_title,"NVL(B.RENT_COUNT,0)","NVL(C.RATING_AVG,0)"
1,14,차근차근 배우는 우리 아이 일상 말하기,97,8.32
2,1706,큰글씨 나는 이렇게 나이들고 싶다,646,9.25165562913907284768211920529801324503
3,1544,"궁금해요, 세종 대왕",692,9.1215469613259668508287292817679558011
4,1065,상실의 시대,902,7.87681159420289855072463768115942028986
5,971,뉴스를 보는 눈,166,9.02040816326530612244897959183673469388
6,3327,중학 국어 기초 완성,24,9.4
7,1873,"일상이 고고학, 나 혼자 국립중앙박물관",0,0
8,2459,수상한 과학,0,0
9,704,임신부·모유수유부의 안전한 약물 사용,19,0
10,1384,2024 SIMPLE 의료법규,0,0


## 표준 조인

ANSI/ISO SQL에서 정의한 조인의 형태

|조인|설명|
|------|-----|
|테이블1 INNER JOIN 테이블2 ON|ON 에서 결합 조건에 해당하는 행들만을 출력|
|테이블1 NATUAL JOIN 테이블2  [USING]| 두 테이블의 모든 동일한 컬럼명의 값이 같을 경우만 출력<br/>USING은 동일한 컬럼명중 비교에 대상을 한정할 경우 사용|
|테이블1 LEFT OUTER JOIN 테이블2 ON|테이블1의 행들은 모두 포함하고 ON을 만족하는 테이블2의 행들을 결합. 테이블2에서의 결합 데이터가 없을 경우 NULL값|
|테이블1 RIGHT OUTER JOIN 테이블2 ON|테이블2의 행들은 모두 포함하고 ON을 만족하는 테이블2의 행들을 결합. 테이블1에서의 결합 데이터가 없을 경우 NULL값|
|테이블1 FULL OUTER JOIN 테이블2 ON|테이블1, 테이블2에서 ON 기준으로 합집합에 해당하는 행들을 포함시켜 결합. 결합 데이터가 없을 경우 NULL값|
|테이블1 CROSS JOIN 테이블2|CATESIAN PRODUCT, 두 개의 테이블을 카테시안 곱을 통해 생성된 행들을 출력|

**[예제 30]** 표준 조인의 여러 연산을 확인합니다.

In [141]:
%%SQL

SELECT * FROM member1 -- member1의 내용을 봅니다.

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길


In [142]:
%%SQL

SELECT * FROM rent WHERE extract(YEAR from rent_date) >= 2023 -- rent 중에서 2023년도 이후의 내역을 가져 옵니다.

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,1954,1,63296p86aoc9,1,2023-02-20 12:01:06,2023-02-26 11:57:41
2,155,1,sn3l7c1fkixu62cl,1,2023-02-20 15:12:09,2023-02-26 09:49:10
3,3230,2,816nhn8itcl5,1,2023-02-20 12:30:17,2023-03-04 09:11:29
4,3193,2,ox48k6f52c,1,2023-02-20 15:51:33,2023-02-26 09:51:10
5,2778,1,hhdkdt9eo5hkac,1,2023-02-20 12:26:41,2023-02-24 16:55:16
...,...,...,...,...,...,...
168506,2339,3,s6vauyvyavx64rb,1,2025-02-08 15:35:36,2025-02-14 15:06:01
168507,774,2,0p7w8fdrbvoi,1,2025-02-08 11:40:19,2025-02-12 13:52:02
168508,137,4,n8g0mcp5rn1q,1,2025-02-08 15:02:42,2025-02-12 16:10:14
168509,938,2,m65gixnyyvie,1,2025-02-08 13:09:13,2025-02-11 09:37:19


- INNER JOIN을 이용하여 member1과 rent를 member_id를 기준으로 결합합니다.

In [143]:
%%SQL

SELECT a.member_id, a.member_name, a.region, a.address1, a. address2, 
    b.book_id, b.stock_seq, b.rent_no, b.rent_date, b.return_date
FROM member1 a INNER JOIN rent b ON a.member_id = b.member_id
WHERE extract(YEAR from rent_date) >= 2023
ORDER BY 1

,member_id,member_name,region,address1,address2,book_id,stock_seq,rent_no,rent_date,return_date
1,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,909,3,1,2024-02-18 12:03:00,2024-02-26 14:31:51
2,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,3444,2,1,2023-03-04 09:47:50,2023-03-07 11:38:57
3,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,1771,2,1,2023-02-24 14:19:33,2023-02-28 13:01:47
4,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,3481,1,1,2024-06-26 14:20:31,2024-07-04 13:09:12
5,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,1807,1,1,2023-05-16 15:00:36,2023-05-24 09:36:23
...,...,...,...,...,...,...,...,...,...,...
62,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,748,1,1,2024-04-14 09:58:39,2024-04-16 10:36:57
63,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,3444,2,1,2024-04-26 14:04:43,2024-05-02 12:17:37
64,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,3161,1,1,2024-09-09 15:17:16,2024-09-16 13:19:51
65,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,2996,2,1,2024-12-08 09:28:53,2024-12-16 10:05:39


- NATURAL JOIN을 이용하여 두 테이블의 동일한 컬럼명을 기준으로 결합합니다.

In [144]:
%%SQL

SELECT member_id, member_name, region, address1, address2, 
    book_id, stock_seq, rent_no, rent_date, return_date
FROM member1 NATURAL JOIN rent
WHERE extract(YEAR from rent_date) >= 2023
ORDER BY 1

,member_id,member_name,region,address1,address2,book_id,stock_seq,rent_no,rent_date,return_date
1,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,909,3,1,2024-02-18 12:03:00,2024-02-26 14:31:51
2,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,3444,2,1,2023-03-04 09:47:50,2023-03-07 11:38:57
3,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,1771,2,1,2023-02-24 14:19:33,2023-02-28 13:01:47
4,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,3481,1,1,2024-06-26 14:20:31,2024-07-04 13:09:12
5,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,1807,1,1,2023-05-16 15:00:36,2023-05-24 09:36:23
...,...,...,...,...,...,...,...,...,...,...
62,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,748,1,1,2024-04-14 09:58:39,2024-04-16 10:36:57
63,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,3444,2,1,2024-04-26 14:04:43,2024-05-02 12:17:37
64,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,3161,1,1,2024-09-09 15:17:16,2024-09-16 13:19:51
65,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,2996,2,1,2024-12-08 09:28:53,2024-12-16 10:05:39


In [145]:
%%SQL

SELECT * FROM rent1 -- rent1 테이블의 내용입니다.

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,2243,2,1jhmhlrlo71brwi4o8,1,NULL,NULL
2,970,1,46vpd6v91zb01e,1,2025-02-20 16:52:29,NULL
3,2403,2,4e1dh553s37,1,2025-02-20 16:43:29,2025-02-24 15:20:23
4,3194,3,94zfx2tf92heh59z,1,NULL,NULL
5,299,1,haejt93chfpl,1,2025-02-20 16:52:14,NULL
6,2239,1,ll65teqf4kzep8l1r,1,2025-02-20 16:56:31,NULL
7,2180,2,nrz2bnr8s6,1,2025-02-20 16:45:01,2025-02-21 12:47:44
8,862,1,r1cgy5kq5bviueq5,1,2025-02-20 16:42:31,2025-02-28 16:28:21
9,3251,2,uce50kz7so3,1,2025-02-20 16:51:15,2025-02-25 10:42:32
10,2364,1,uck1pr4ofirj2a513,1,2025-02-20 16:46:03,2025-02-28 12:50:36


- LEFT OUTER JOIN을 이용하여 좌측의 테이블을 기준으로 우측의 테이블을 결합합니다. 우측 테이블에 내용일 없을 경우 NULL 값이 들어 갑니다.

In [146]:
%%SQL
SELECT * FROM member1

,member_id,member_name,region,address1,address2
1,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길
3,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길
5,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길
6,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길
7,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길
8,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길


In [147]:
%%SQL
SELECT * FROM rent1 

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,2243,2,1jhmhlrlo71brwi4o8,1,NULL,NULL
2,970,1,46vpd6v91zb01e,1,2025-02-20 16:52:29,NULL
3,2403,2,4e1dh553s37,1,2025-02-20 16:43:29,2025-02-24 15:20:23
4,3194,3,94zfx2tf92heh59z,1,NULL,NULL
5,299,1,haejt93chfpl,1,2025-02-20 16:52:14,NULL
6,2239,1,ll65teqf4kzep8l1r,1,2025-02-20 16:56:31,NULL
7,2180,2,nrz2bnr8s6,1,2025-02-20 16:45:01,2025-02-21 12:47:44
8,862,1,r1cgy5kq5bviueq5,1,2025-02-20 16:42:31,2025-02-28 16:28:21
9,3251,2,uce50kz7so3,1,2025-02-20 16:51:15,2025-02-25 10:42:32
10,2364,1,uck1pr4ofirj2a513,1,2025-02-20 16:46:03,2025-02-28 12:50:36


In [148]:
%%SQL

SELECT a.member_id, a.member_name, a.region, a.address1, a. address2, 
    b.book_id, b.stock_seq, b.rent_no, b.rent_date, b.return_date
FROM member1 a LEFT OUTER JOIN rent1 b ON a.member_id = b.member_id
ORDER BY 1

,member_id,member_name,region,address1,address2,book_id,stock_seq,rent_no,rent_date,return_date
1,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,NULL,NULL,NULL,NULL,NULL
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길,NULL,NULL,NULL,NULL,NULL
3,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길,NULL,NULL,NULL,NULL,NULL
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길,NULL,NULL,NULL,NULL,NULL
5,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길,NULL,NULL,NULL,NULL,NULL
6,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길,NULL,NULL,NULL,NULL,NULL
7,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길,NULL,NULL,NULL,NULL,NULL
8,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,NULL,NULL,NULL,NULL,NULL


- RIGHT OUTER JOIN을 이용하여 우측의 테이블을 기준으로 좌측의 테이블을 결합합니다. 좌측 테이블에 내용일 없을 경우 NULL 값이 들어 갑니다.

In [149]:
%%SQL

SELECT a.member_id, a.member_name, a.region, a.address1, a. address2, 
    b.book_id, b.stock_seq, b.rent_no, b.rent_date, b.return_date
FROM member1 a RIGHT OUTER JOIN rent1 b ON a.member_id = b.member_id
ORDER BY 1

,member_id,member_name,region,address1,address2,book_id,stock_seq,rent_no,rent_date,return_date
1,NULL,NULL,NULL,NULL,NULL,2239,1,1,2025-02-20 16:56:31,NULL
2,NULL,NULL,NULL,NULL,NULL,970,1,1,2025-02-20 16:52:29,NULL
3,NULL,NULL,NULL,NULL,NULL,299,1,1,2025-02-20 16:52:14,NULL
4,NULL,NULL,NULL,NULL,NULL,3251,2,1,2025-02-20 16:51:15,2025-02-25 10:42:32
5,NULL,NULL,NULL,NULL,NULL,2243,2,1,NULL,NULL
6,NULL,NULL,NULL,NULL,NULL,2180,2,1,2025-02-20 16:45:01,2025-02-21 12:47:44
7,NULL,NULL,NULL,NULL,NULL,2403,2,1,2025-02-20 16:43:29,2025-02-24 15:20:23
8,NULL,NULL,NULL,NULL,NULL,862,1,1,2025-02-20 16:42:31,2025-02-28 16:28:21
9,NULL,NULL,NULL,NULL,NULL,3194,3,1,NULL,NULL
10,NULL,NULL,NULL,NULL,NULL,2364,1,1,2025-02-20 16:46:03,2025-02-28 12:50:36


- FULL OUTER JOIN을 이용하여 양측의 모두를 기준으로 member_id를 기준으로 합집합 연산을 하여 구성합니다.

In [150]:
%%SQL

SELECT a.member_id, a.member_name, a.region, a.address1, a. address2, 
    b.book_id, b.stock_seq, b.rent_no, b.rent_date, b.return_date
FROM member1 a FULL OUTER JOIN rent1 b ON a.member_id = b.member_id
ORDER BY 1

,member_id,member_name,region,address1,address2,book_id,stock_seq,rent_no,rent_date,return_date
1,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,NULL,NULL,NULL,NULL,NULL
2,9005r5vmi,변채원,서울특별시,성북구,월계로 7길,NULL,NULL,NULL,NULL,NULL
3,kijtmcsxqvxe4d,남채원,경상북도,포항시 남구,서원재로 8번길,NULL,NULL,NULL,NULL,NULL
4,n3uo4sgnjs55rx,김연우,경상북도,포항시 북구,학산로 1번길,NULL,NULL,NULL,NULL,NULL
5,nnvy9vtclgyfb,최승우,경기도,양평군,망능길 3번길,NULL,NULL,NULL,NULL,NULL
6,osfi8te4nv9m,권예은,경기도,양평군,신촌길 2번길,NULL,NULL,NULL,NULL,NULL
7,q6bl2tdplxz,변태진,충청북도,청주시 흥덕구,경산로 6번길,NULL,NULL,NULL,NULL,NULL
8,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,NULL,NULL,NULL,NULL,NULL
9,NULL,NULL,NULL,NULL,NULL,2243.0,2.0,1.0,NULL,NULL
10,NULL,NULL,NULL,NULL,NULL,3194.0,3.0,1.0,NULL,NULL


- 위의 예제에서는 교집합인 경우는 없었는데요, 교집합이 있는 경우의 출력을 확인합니다.

In [151]:
%%SQL
SELECT * FROM rent

,book_id,stock_seq,member_id,rent_no,rent_date,return_date
1,3466,1,e3me1m5mom0g1n56,1,2019-10-02 09:19:54,2019-10-06 13:05:55
2,863,2,nbfn8zzuwjqym,1,2019-10-02 15:25:31,2019-10-05 12:03:37
3,3369,1,qmgx9763k09f3,1,2019-10-02 11:56:35,2019-10-09 13:01:31
4,382,1,012ywcryvy,1,2019-10-02 16:28:40,2019-10-07 13:43:02
5,3328,1,qjms3d4nqba8,1,2019-10-02 12:55:05,2019-10-04 10:01:03
...,...,...,...,...,...,...
415436,2339,3,s6vauyvyavx64rb,1,2025-02-08 15:35:36,2025-02-14 15:06:01
415437,774,2,0p7w8fdrbvoi,1,2025-02-08 11:40:19,2025-02-12 13:52:02
415438,137,4,n8g0mcp5rn1q,1,2025-02-08 15:02:42,2025-02-12 16:10:14
415439,938,2,m65gixnyyvie,1,2025-02-08 13:09:13,2025-02-11 09:37:19


In [152]:
%%SQL

SELECT a.member_id, a.member_name, a.region, a.address1, a. address2, 
    b.book_id, b.stock_seq, b.rent_no, b.rent_date, b.return_date
FROM member1 a LEFT OUTER JOIN rent b 
    ON a.member_id = b.member_id
ORDER BY 1

,member_id,member_name,region,address1,address2,book_id,stock_seq,rent_no,rent_date,return_date
1,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,3201,1,1,2020-10-20 13:37:25,2020-10-27 13:25:43
2,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,3481,1,1,2024-06-26 14:20:31,2024-07-04 13:09:12
3,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,3411,3,1,2021-07-25 09:21:24,2021-07-31 09:58:37
4,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,37,1,1,2021-09-01 10:47:28,2021-09-09 09:22:38
5,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,857,4,1,2022-09-05 10:12:27,2022-09-09 13:30:37
...,...,...,...,...,...,...,...,...,...,...
151,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,748,1,1,2024-04-14 09:58:39,2024-04-16 10:36:57
152,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,3444,2,1,2024-04-26 14:04:43,2024-05-02 12:17:37
153,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,3161,1,1,2024-09-09 15:17:16,2024-09-16 13:19:51
154,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,2996,2,1,2024-12-08 09:28:53,2024-12-16 10:05:39


- CROSS JOIN으로 두 테이블에서 조합가능한 모든 경우를 출력합니다.

In [153]:
%%SQL

SELECT a.member_id, a.member_name, a.region, a.address1, a. address2, 
    b.book_id, b.stock_seq, b.rent_no, b.rent_date, b.return_date
FROM member1 a, rent1 b
WHERE a.member_id = b.member_id
ORDER BY 1

,member_id,member_name,region,address1,address2,book_id,stock_seq,rent_no,rent_date,return_date


In [154]:
%%SQL

SELECT a.member_id, a.member_name, a.region, a.address1, a. address2, 
    b.book_id, b.stock_seq, b.rent_no, b.rent_date, b.return_date
FROM member1 a CROSS JOIN rent1 b
ORDER BY 1

,member_id,member_name,region,address1,address2,book_id,stock_seq,rent_no,rent_date,return_date
1,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,970,1,1,2025-02-20 16:52:29,NULL
2,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,299,1,1,2025-02-20 16:52:14,NULL
3,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,3251,2,1,2025-02-20 16:51:15,2025-02-25 10:42:32
4,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,2364,1,1,2025-02-20 16:46:03,2025-02-28 12:50:36
5,71r05w32wx4e5j,최다현,충청남도,아산시,아산만로 9번길,2239,1,1,2025-02-20 16:56:31,NULL
...,...,...,...,...,...,...,...,...,...,...
76,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,2180,2,1,2025-02-20 16:45:01,2025-02-21 12:47:44
77,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,2403,2,1,2025-02-20 16:43:29,2025-02-24 15:20:23
78,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,862,1,1,2025-02-20 16:42:31,2025-02-28 16:28:21
79,zcitby9bvw,이서진,제주특별자치도,제주시,수산 6길,3194,3,1,NULL,NULL


## Self-Join

1.
```SQL
SELECT ALIAS명1.컬럼명, ALIAS명2.컬럼명
FROM 테이블 ALIAS명1, 테이블 ALIAS명2
WHERE ALIAS명2.컬럼명 = ALIAS명1.컬럼명;
```
2.
```SQL
SELECT ALIAS명1.컬럼명, ALIAS명2.컬럼명
FROM 테이블 ALIAS명1 {INNER JOIN | {LEFT|RIGHT|FULL} OUTER JOIN} 테이블 ALIAS명2
ON ALIAS명2.컬럼명 = ALIAS명1.컬럼명;
```

**[예제 31]** Self Join을 가지고 계층적 구조를 지닌 테이블의 내용을 조회합니다.

In [155]:
%%SQL

SELECT * 
FROM BOOKCATEGORY -- 계층 구조를 지닌 테이블 입니다. parent_categ_id 는 레코드의 상위 카테고리의 id가 들어 있습니다.

,categ_id,categ_name,parent_categ_id,reg_date,use_yn,nav_seq
1,309,언어학교재,300.0,2019-09-18 10:58:20,1,9
2,310,기타 외국문학,300.0,2019-09-20 12:13:48,1,10
3,312,가정학/소비자학,311.0,2019-09-23 11:34:20,1,1
4,313,국악/성악/음대 교재,311.0,2019-09-23 12:14:33,1,2
5,314,디자인/미술/만화,311.0,2019-09-25 11:19:00,1,4
...,...,...,...,...,...,...
1602,1595,미국교과서,1588.0,2025-02-13 15:14:13,1,7
1603,1601,초등학습법/플래너,1600.0,2025-02-19 13:16:07,1,2
1604,1602,학습교구/퍼즐/게임,1600.0,2025-02-17 12:15:16,1,1
1605,1604,EBS 중학교,1603.0,2025-02-18 11:05:00,1,1


- parent_categ_id가 NULL인 경우 즉 상위 카테고리가 없은 경우를 가져옵니다.

In [156]:
%%SQL

SELECT * FROM BOOKCATEGORY WHERE parent_categ_id is NULL

,categ_id,categ_name,parent_categ_id,reg_date,use_yn,nav_seq
1,1,가정 살림,NULL,2018-05-03 16:17:09,1,1
2,41,건강 취미,NULL,2018-05-08 10:11:15,1,2
3,116,경제 경영,NULL,2018-05-08 16:12:00,1,3
4,161,국어 외국어 사전,NULL,2018-05-08 16:34:40,1,4
5,237,대학교재,NULL,2018-05-11 09:08:10,1,5
6,351,만화/라이트노벨,NULL,2018-05-11 16:18:31,1,6
7,404,사회 정치,NULL,2018-05-14 09:55:54,1,7
8,479,소설/시/희곡,NULL,2018-05-15 14:47:44,1,8
9,537,수험서 자격증,NULL,2018-05-17 09:47:46,1,9
10,721,어린이,NULL,2018-05-24 16:49:44,1,14


- INNER JOIN 좌, 우측 테이블 모두 BOOKCATEGORY를 사용하고 결합 기준을 categ_id와 parent_categ_id를 사용하여 레코드의 상위 카테고리를 가져 옵니다.

In [157]:
%%SQL

SELECT a.categ_name as 대분류, b.categ_name as 소분류
FROM BOOKCATEGORY a INNER JOIN BOOKCATEGORY b ON a.categ_id = b.parent_categ_id

,대분류,소분류
1,어문학 계열,언어학교재
2,어문학 계열,기타 외국문학
3,예체능/문화/기타 계열,가정학/소비자학
4,예체능/문화/기타 계열,국악/성악/음대 교재
5,예체능/문화/기타 계열,디자인/미술/만화
...,...,...
1575,영어,미국교과서
1576,초등 학습자료/교구,초등학습법/플래너
1577,초등 학습자료/교구,학습교구/퍼즐/게임
1578,중등참고서,EBS 중학교


In [158]:
%%SQL
SELECT * FROM BOOKCATEGORY

,categ_id,categ_name,parent_categ_id,reg_date,use_yn,nav_seq
1,309,언어학교재,300.0,2019-09-18 10:58:20,1,9
2,310,기타 외국문학,300.0,2019-09-20 12:13:48,1,10
3,312,가정학/소비자학,311.0,2019-09-23 11:34:20,1,1
4,313,국악/성악/음대 교재,311.0,2019-09-23 12:14:33,1,2
5,314,디자인/미술/만화,311.0,2019-09-25 11:19:00,1,4
...,...,...,...,...,...,...
1602,1595,미국교과서,1588.0,2025-02-13 15:14:13,1,7
1603,1601,초등학습법/플래너,1600.0,2025-02-19 13:16:07,1,2
1604,1602,학습교구/퍼즐/게임,1600.0,2025-02-17 12:15:16,1,1
1605,1604,EBS 중학교,1603.0,2025-02-18 11:05:00,1,1


In [159]:
%%SQL

SELECT a.categ_name as 대분류, b.categ_name as 소분류
FROM BOOKCATEGORY a LEFT OUTER JOIN BOOKCATEGORY b ON a.categ_id = b.parent_categ_id

,대분류,소분류
1,어문학 계열,언어학교재
2,어문학 계열,기타 외국문학
3,예체능/문화/기타 계열,가정학/소비자학
4,예체능/문화/기타 계열,국악/성악/음대 교재
5,예체능/문화/기타 계열,디자인/미술/만화
...,...,...
2951,퍼즐/스도쿠,NULL
2952,수학교육,NULL
2953,5-6학년 놀이,NULL
2954,중국어 입문서,NULL


## 계층형 질의 - Oracle DBMS

```SQL
SELECT …
FROM 테이블
[WHERE 조건문]
START WITH 조건문
CONNECT BY [NO CYCLE]  PRIOR 조건문
[ORDER SIBLINGS BY 컬럼1 또는 순서, … ]
```

**[예제 31]** 계층적 질의를 활용하여 재귀적으로 계측 구조를 가져오는 기능을 활용합니다.

- START WITH 에서는 첫번째 뎁스의 내용을 정의합니다: parent_categ_id 이 NULL인 것이니 상위 카테고리가 없는 것입니다. 즉 최상위 구조입니다.
- CONNECT BY 에서는 START WITH에서 추출한 내용 다음에 가져올 조건을 정의합니다. PRIOR 키워드가 있는 쪽이 상위구조의 컬럼입니다.

  여기서는 parent_categ_id가 상위 구조의 categ_id와 같으면 다음 단계 출력을로 가져오는데요. CONNECT BY에 해당하는 행이 없을 때까지 가져옵니다.
- ORDER SIBLING BY 에서는 이후에 가져오는 행들의 정렬의 기준입니다. 


In [160]:
%%SQL

SELECT categ_id as 카테고리ID, categ_name as 카테고리명, nvl(parent_categ_id, 0) as 상위카테고리ID, nav_seq, LEVEL
FROM BOOKCATEGORY
START WITH parent_categ_id is NULL
CONNECT BY PRIOR categ_id = parent_categ_id
ORDER SIBLINGS BY nav_seq	

,카테고리ID,카테고리명,상위카테고리ID,nav_seq,LEVEL
1,1,가정 살림,0,1,1
2,2,임신/출산,1,1,2
3,3,임신,2,1,3
4,4,태교,2,2,3
5,11,자녀교육,1,2,2
...,...,...,...,...,...
1602,1598,영재교육원대비,1532,15,2
1603,1599,월간지,1532,16,2
1604,1600,초등 학습자료/교구,1532,17,2
1605,1602,학습교구/퍼즐/게임,1600,1,3


In [161]:
%%SQL

SELECT categ_id as 카테고리ID, categ_name as 카테고리명, nvl(parent_categ_id, 0) as 상위카테고리ID, nav_seq
FROM BOOKCATEGORY
START WITH parent_categ_id is NULL-- AND categ_id = 1
CONNECT BY parent_categ_id = PRIOR categ_id
ORDER SIBLINGS BY nav_seq

,카테고리ID,카테고리명,상위카테고리ID,nav_seq
1,1,가정 살림,0,1
2,2,임신/출산,1,1
3,3,임신,2,1
4,4,태교,2,2
5,11,자녀교육,1,2
...,...,...,...,...
1602,1598,영재교육원대비,1532,15
1603,1599,월간지,1532,16
1604,1600,초등 학습자료/교구,1532,17
1605,1602,학습교구/퍼즐/게임,1600,1


- SYS_CONNECT_BY_PATH를 사용하면 지금까지 자신을 포함한 탐색과정에 있던 CATEG_NAME을 가져옵니다.

In [162]:
%%SQL

SELECT categ_id as 카테고리ID, categ_name as 카테고리명, nvl(parent_categ_id, 0) as 상위카테고리ID, LEVEL, SYS_CONNECT_BY_PATH(categ_name, '|')
FROM BOOKCATEGORY
START WITH parent_categ_id is NULL AND categ_id = 1
CONNECT BY parent_categ_id = PRIOR categ_id
ORDER SIBLINGS BY nav_seq

,카테고리ID,카테고리명,상위카테고리ID,LEVEL,"SYS_CONNECT_BY_PATH(CATEG_NAME,'|')"
1,1,가정 살림,0,1,|가정 살림
2,2,임신/출산,1,2,|가정 살림|임신/출산
3,3,임신,2,3,|가정 살림|임신/출산|임신
4,4,태교,2,3,|가정 살림|임신/출산|태교
5,11,자녀교육,1,2,|가정 살림|자녀교육
6,13,놀이교육,11,3,|가정 살림|자녀교육|놀이교육
7,14,독서교육,11,3,|가정 살림|자녀교육|독서교육
8,16,감성/체험교육,11,3,|가정 살림|자녀교육|감성/체험교육
9,15,영어교육,11,3,|가정 살림|자녀교육|영어교육
10,12,학습법일반,11,3,|가정 살림|자녀교육|학습법일반
